# SOIC major revision (4492-SOIC): all remaining experiments in one notebook

**Paper:** *Detection and Classification of Apple Diseases using ML Techniques: A Leakage-Free Deep-Learning Framework with Honest Cross-Dataset Evaluation*

This notebook replaces the separate scripts `missing_experiments/01..08` and the Colab notebook. Everything is here, cell by cell.

**What changed compared with the old scripts (Phase 1 fixes):**

| # | Problem in the old code | Fix in this notebook |
|---|---|---|
| 1 | Colab disconnects lost work; only whole runs were saved | Every training run saves a checkpoint **after every epoch** and resumes from it. Every finished run is written to a JSON file immediately |
| 2 | Data-efficiency (03) used the *full* FGVC8 folder (14,491 images, so 5% = 492 images), not the paper's balanced 500/class pool (5% = 80 images) | A fixed, balanced **500 per class** FGVC8 pool is built once and saved as a CSV, then reused everywhere. ResNet50 is re-run with the same code as a reference |
| 3 | The model definitions did not match how the paper checkpoints were saved (DenseNet ReLU, InceptionV3 layout, VGG16 head), so full checkpoints could not be loaded for DenseNet/Inception/VGG | Models are rebuilt with the **exact** `nn.Sequential` layout of the saved checkpoints; all 5 load with `strict=True` |
| 4 | Leaky split (02): the base-leaf regex missed `_newNNdegFlipLR`, `_newPixelN`, `_newGRR`, so the overlap was under-counted; the model was not saved and not tested on field data | Correct regex (all 9,819 files map to 3,277 leaves); the model is saved and evaluated on (a) the leaky test set, (b) its clean (never-seen-leaf) part, (c) PlantDoc and FGVC8 zero-shot. An honest model is trained with the **same code** for a fair comparison |
| 5 | Domain adaptation (07) used a different field pool and a 30% split, so it could not be compared with Table 9 | Uses the same 500/class pool and the **same 400-image field test split** as the data-efficiency study, plus a *source-only* baseline |
| 6 | GAN images were added without checking them; many look like healthy leaves | Synthetic images are kept only if they are not near-duplicates **and** the paper classifier calls them cedar-apple-rust with confidence ≥ 0.9. The baseline is re-trained with the same code |
| 7 | t-SNE figure was never saved | Figure saved (PNG + PDF), plus a FGVC8-scab control, an ImageNet-feature view, and an image panel |
| 8 | Nothing for field error analysis / confidence intervals | New section: confusion matrices, bootstrap 95% CIs, most-confident errors, and accuracy vs brightness / blur / vegetation fraction |
| 9 | CAM-in-leaf could not be validated without hand-drawn masks | New: an **independent GrabCut leaf mask** on all 490 test images + optional LabelMe manual masks |

**Run order.** Top to bottom. Every section has a `RUN[...]` switch in the CONFIG cell. Sections 2–6 are inference only (fine on a laptop CPU). Sections 7–12 train networks (slow on CPU, so use a GPU or Colab; they resume automatically).

**Tip.** First run the whole notebook once with `FAST_DEV_RUN = True` (a few images, 1 epoch, a couple of minutes). If that finishes without errors, set it back to `False`.

## 0. Installation (one time)

**Local (VS Code on Windows):**
1. Install Python 3.10 or 3.11 and the VS Code *Python* + *Jupyter* extensions.
2. Open a terminal in the `code soic` folder and create an environment:
   ```
   python -m venv .venv
   .venv\Scripts\activate
   ```
3. Install PyTorch. CPU build:
   ```
   pip install torch torchvision --index-url https://download.pytorch.org/whl/cpu
   ```
   Your NVIDIA GPU is only useful if a CUDA build of PyTorch still supports it. You can try `--index-url https://download.pytorch.org/whl/cu118`; the device cell below tells you whether the GPU really works and otherwise falls back to the CPU.
4. Install the rest:
   ```
   pip install numpy scipy scikit-learn opencv-python pillow matplotlib pandas tqdm ipykernel
   pip install ultralytics   # only for the optional YOLO re-training (Section 7)
   pip install labelme       # only for the optional manual masks (Section 6)
   ```
5. In VS Code choose the `.venv` kernel (top right of the notebook).

**Colab:** set `RUN_ENV = "colab"` in the next cell. The cell mounts Drive and unzips `colab_bundle.zip`. All results are written to Drive after every epoch, so a disconnect loses at most one epoch.

In [ ]:
# 0.1  Optional: install packages from inside the notebook (uncomment if needed)
# %pip install numpy scipy scikit-learn opencv-python pillow matplotlib pandas tqdm
# %pip install ultralytics

## 1. CONFIG: the only cell you normally edit

In [ ]:
# 1.1  CONFIG -------------------------------------------------------------------------
import os, sys, platform, json, time, glob, re, random, shutil, hashlib, zipfile, math, contextlib, warnings, csv
from collections import OrderedDict, defaultdict, Counter
warnings.filterwarnings("ignore", category=UserWarning)

RUN_ENV = "local"        # "local"  = VS Code on your PC  |  "colab" = Google Colab
FAST_DEV_RUN = False     # True = tiny smoke test (few images, 1 epoch) to check the notebook runs end-to-end

# Which sections to run (True/False). Inference sections are cheap; training sections are heavy.
RUN = dict(
    data_prep      = True,   # 2. build/check file lists, balanced FGVC8 pools, image cache  (needed by everything)
    sanity         = True,   # 3. reproduce paper numbers + field error analysis + bootstrap CIs + CPU latency
    tsne           = True,   # 4. Frogeye vs Black-Rot feature-space check (Reviewer A Q1 / W2)
    gradcam        = True,   # 5. per-class CAM-in-leaf + independent GrabCut mask (Reviewer A Q5, Reviewer B)
    manual_masks   = True,   # 6. prepare / score LabelMe manual masks (Reviewer B) - skips if no masks drawn yet
    yolo_import    = True,   # 7. import the finished YOLOv8-vs-YOLOv11 results (Reviewer A Q3)
    yolo_train     = False,  # 7b. re-train YOLO (optional, needs ultralytics; slow on CPU)
    data_eff       = True,   # 8.  TRAINING: data-efficiency for more backbones (Reviewer A W5, Reviewer B)
    leaky          = True,   # 9.  TRAINING: leaky vs honest split + field evaluation (Reviewer A W3)
    domain_adapt   = True,   # 10. TRAINING: source-only / CORAL / DANN baselines (Reviewer B)
    gan            = True,   # 11. TRAINING: GAN oversampling of cedar apple rust (Reviewer A Q4)
    aug_ablation   = False,  # 12. TRAINING (optional): which augmentation group matters (Reviewer A Q2)
    five_seeds     = False,  # 13. TRAINING (optional, very heavy): 5 seeds x 5 backbones (Reviewer B)
)

# ---- experiment settings (paper protocol) ----
SEEDS            = [0, 1, 2]
DE_BACKBONES     = ["resnet50", "mobilenet_v2", "densenet121"]  # resnet50 = same-code reference for Table 9
DE_FRACTIONS     = [0.05, 0.10, 0.25, 0.50, 1.00]
DE_INCLUDE_LEAKY_INIT = True       # also start ResNet50 from the LEAKY backbone (needs Section 9 first)
FGVC8_POOL_EVAL  = 700             # images/class for the zero-shot evaluation set (paper: 700/class = 2,800)
FGVC8_POOL_DE    = 500             # images/class for data-efficiency + domain adaptation (paper: 500/class)
FIELD_TEST_FRAC  = 0.20            # fixed held-out field test split (paper: 20%)
FIELD_SPLIT_SEED = 1234
DA_METHODS       = ["source_only", "coral", "dann"]
DA_EPOCHS        = 20
GAN_EPOCHS       = 2000
GAN_N_GENERATE   = 300
GAN_MIN_CONF     = 0.90            # keep synthetic images the paper classifier calls cedar rust with p >= 0.90
GAN_SIM_THRESH   = 0.98            # flag synthetic images with cosine similarity >= 0.98 to a real image
AUG_SEEDS        = [0]
FIVE_SEEDS       = [0, 1, 2, 3, 4]
BOOTSTRAP_N      = 2000
CACHE_MAX_SIDE   = 384             # large field photos (4000x2672) are shrunk ONCE to this size -> much faster

# ---- training protocol (Section 2.4 of the paper) ----
TRAIN_CFG = dict(phase1_epochs=5, phase1_lr=1e-3, phase2_epochs=25, phase2_lr=1e-4,
                 weight_decay=1e-4, patience=8, batch_size=16)

# ---- paths ----
if RUN_ENV == "local":
    # VS Code starts the kernel in the notebook's folder ("code soic"). Change this line if it does not.
    CODE_ROOT    = r"D:\co work\SOIC major revision\code soic"
    PROJECT_ROOT = os.path.join(CODE_ROOT, "1")                 # data, checkpoints, cache, manual masks live here
    DATA_ROOT    = os.path.join(PROJECT_ROOT, "data")
    CKPT_ROOT    = os.path.join(PROJECT_ROOT, "checkpoints")
    CACHE_ROOT   = os.path.join(PROJECT_ROOT, "data_cache")
    RESULTS_ROOT = os.path.join(CODE_ROOT, "v2 result")         # all v2 results go here
else:
    from google.colab import drive
    drive.mount("/content/drive")
    ZIP_PATH     = "/content/drive/MyDrive/colab_bundle.zip"
    PROJECT_ROOT = "/content/soic_bundle"
    if not os.path.isdir(os.path.join(PROJECT_ROOT, "data")):
        os.makedirs(PROJECT_ROOT, exist_ok=True)
        with zipfile.ZipFile(ZIP_PATH) as zf:
            zf.extractall(PROJECT_ROOT)
    DATA_ROOT    = os.path.join(PROJECT_ROOT, "data")
    CKPT_ROOT    = os.path.join(PROJECT_ROOT, "checkpoints")
    CACHE_ROOT   = "/content/data_cache"
    RESULTS_ROOT = "/content/drive/MyDrive/soic_result_new"   # on Drive -> survives disconnects

if FAST_DEV_RUN:
    RESULTS_ROOT = RESULTS_ROOT + "_devrun"   # never mix smoke-test numbers with real ones
    SEEDS, AUG_SEEDS, FIVE_SEEDS = [0], [0], [0]
    DE_FRACTIONS = [0.5, 1.0]
    DE_BACKBONES = ["resnet50", "mobilenet_v2"]
    DA_EPOCHS, GAN_EPOCHS, GAN_N_GENERATE, BOOTSTRAP_N = 1, 2, 8, 50
    FGVC8_POOL_EVAL, FGVC8_POOL_DE = 12, 10
    TRAIN_CFG.update(phase1_epochs=1, phase2_epochs=1)
DEV_PER_CLASS = 6   # images per class used when FAST_DEV_RUN is True

PV_ROOT       = os.path.join(DATA_ROOT, "plantvillage_leakfree")    # train/val/test/<class>
LEAKY_ROOT    = os.path.join(DATA_ROOT, "plantvillage_raw_leaky")   # <class>/ (9,819 files incl. on-disk copies)
FGVC8_ROOT    = os.path.join(DATA_ROOT, "fgvc8_by_class")           # <class>/ (black_rot = frogeye leaf spot)
PLANTDOC_ROOT = os.path.join(DATA_ROOT, "plantdoc_external")        # <class>/ (no black_rot)
YOLO_ROOT     = os.path.join(DATA_ROOT, "yolo_plantdoc")
SPLITS_DIR    = os.path.join(RESULTS_ROOT, "splits")                # fixed CSV lists (keep them!)
FIG_DIR       = os.path.join(RESULTS_ROOT, "figures")
MANUAL_DIR    = os.path.join(PROJECT_ROOT, "manual_annotations")
CKPT_FILES = {"resnet50": "cls_resnet50.pt", "mobilenet_v2": "cls_mobilenetv2.pt",
              "densenet121": "cls_densenet121.pt", "vgg16": "cls_vgg16.pt", "inception_v3": "cls_inceptionv3.pt"}
_ZIP_DIR = CODE_ROOT if RUN_ENV == "local" else PROJECT_ROOT
OLD_RESULT_ZIPS = [os.path.join(_ZIP_DIR, "soic_results (1).zip"), os.path.join(_ZIP_DIR, "soic_results (2).zip")]

CLASSES = ["apple_scab", "black_rot", "cedar_apple_rust", "healthy"]   # fixed label order used everywhere
CLASS_TITLES = ["Apple scab", "Black rot", "Cedar apple rust", "Healthy"]
IS_WINDOWS = platform.system() == "Windows"
NUM_WORKERS = 0 if (IS_WINDOWS or RUN_ENV == "local") else 2   # 0 is the safe choice inside Jupyter on Windows

for d in [RESULTS_ROOT, SPLITS_DIR, FIG_DIR, CACHE_ROOT]:
    os.makedirs(d, exist_ok=True)

# quick check that the folders exist
for name, p in [("DATA_ROOT", DATA_ROOT), ("CKPT_ROOT", CKPT_ROOT), ("PV_ROOT", PV_ROOT), ("LEAKY_ROOT", LEAKY_ROOT),
                ("FGVC8_ROOT", FGVC8_ROOT), ("PLANTDOC_ROOT", PLANTDOC_ROOT)]:
    print(f"{name:14s} {'OK     ' if os.path.exists(p) else 'MISSING'}  {p}")
print("RESULTS_ROOT  ", RESULTS_ROOT)
assert os.path.isdir(PV_ROOT), "PlantVillage folder not found: open the notebook from the 'code soic' folder or fix PROJECT_ROOT."

## 1.2 Device check: does the GPU really work?

PyTorch may report a GPU that cannot be used (too old for the installed build, or too little memory). This cell runs a real convolution on the GPU. If that fails, the notebook uses the CPU. With little GPU memory the batch is split into smaller *micro-batches* with gradient accumulation, so the effective batch size stays 16 as in the paper.

In [ ]:
# 1.2  Device ----------------------------------------------------------------------------
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision as tv
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from PIL import Image, ImageDraw
import cv2
import matplotlib
import matplotlib.pyplot as plt
try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(x, **k): return x
%matplotlib inline

def pick_device():
    if torch.cuda.is_available():
        try:
            name = torch.cuda.get_device_name(0)
            x = torch.randn(2, 3, 64, 64, device="cuda")
            w = torch.randn(8, 3, 3, 3, device="cuda")
            _ = F.conv2d(x, w).sum().item()
            mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
            print(f"GPU works: {name}  ({mem:.1f} GB)")
            return torch.device("cuda"), mem
        except Exception as e:
            print("A GPU is visible but NOT usable with this PyTorch build -> using CPU.\n   reason:", str(e)[:200])
    return torch.device("cpu"), 0.0

DEVICE, GPU_MEM_GB = pick_device()
USE_AMP = DEVICE.type == "cuda"
if DEVICE.type == "cpu":
    torch.set_num_threads(max(1, os.cpu_count() or 1))
    MICRO_BATCH = TRAIN_CFG["batch_size"]
else:
    MICRO_BATCH = 16 if GPU_MEM_GB >= 6 else (8 if GPU_MEM_GB >= 3.5 else 4)
ACCUM_STEPS = max(1, TRAIN_CFG["batch_size"] // MICRO_BATCH)
EVAL_BATCH = 32 if DEVICE.type == "cuda" and GPU_MEM_GB >= 3.5 else 16
print(f"device={DEVICE}  AMP={USE_AMP}  micro-batch={MICRO_BATCH} x {ACCUM_STEPS} accumulation "
      f"= effective batch {MICRO_BATCH*ACCUM_STEPS}  | torch {torch.__version__}, torchvision {tv.__version__}, "
      f"python {platform.python_version()}")

def make_scaler():
    try:
        return torch.amp.GradScaler("cuda", enabled=USE_AMP)
    except Exception:
        return torch.cuda.amp.GradScaler(enabled=USE_AMP)

def autocast_ctx():
    if USE_AMP:
        return torch.autocast(device_type="cuda", dtype=torch.float16)
    return contextlib.nullcontext()

SMALL_GPU = DEVICE.type == "cuda" and GPU_MEM_GB < 3.5
CAM_BATCH = MICRO_BATCH if DEVICE.type == "cuda" else 16   # Grad-CAM keeps activations for backward -> training-sized memory

def is_oom(e):
    return isinstance(e, RuntimeError) and "out of memory" in str(e).lower()

OOM_SKIPPED = []
def oom_guard(tag, fn, *a, **k):
    # Runs one training job. If the GPU runs out of memory, the job is logged and skipped
    # (its last finished epoch stays saved), instead of stopping the whole notebook.
    try:
        return fn(*a, **k)
    except RuntimeError as e:
        if not is_oom(e):
            raise
        OOM_SKIPPED.append(tag)
        print(f"!! {tag}: GPU out of memory -> skipped (run it later on a bigger GPU / Colab)")
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()
        return None

if SMALL_GPU:
    print("NOTE: small GPU. Training uses micro-batches of", MICRO_BATCH,
          "with gradient accumulation (effective batch 16), but BatchNorm still sees", MICRO_BATCH,
          "images at a time, which differs from the paper (16). Numbers for the final paper are better produced on a >= 8 GB GPU (e.g. Colab T4).")

## 1.3 Shared library: data, models, resumable training, evaluation

Everything that the experiments share. Nothing here needs editing.

In [ ]:
# 1.3a  small I/O helpers: atomic saves (a crash never leaves a half-written file) --------------
def _json_default(o):
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.floating,)): return float(o)
    if isinstance(o, np.ndarray): return o.tolist()
    if torch.is_tensor(o): return o.tolist()
    return str(o)

def save_json(obj, path):
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    tmp = path + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, default=_json_default)
    os.replace(tmp, path)

def load_json(path, default=None):
    if not os.path.exists(path):
        return default
    with open(path, encoding="utf-8") as f:
        return json.load(f)

def torch_save_atomic(obj, path):
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    tmp = path + ".tmp"
    torch.save(obj, tmp)
    os.replace(tmp, path)

def torch_load(path):
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:   # older torch has no weights_only argument
        return torch.load(path, map_location="cpu")

LOG_PATH = os.path.join(RESULTS_ROOT, "run_log.txt")
def log(*args):
    msg = " ".join(str(a) for a in args)
    print(msg)
    with open(LOG_PATH, "a", encoding="utf-8") as f:
        f.write(time.strftime("%Y-%m-%d %H:%M:%S ") + msg + "\n")

class ResultStore:
    # A JSON list of finished runs. A run is appended the moment it finishes, so nothing is lost.
    def __init__(self, path):
        self.path = path
        self.rows = load_json(path, [])
    def _match(self, r, key):
        return all(r.get(k) == v for k, v in key.items())
    def has(self, **key):
        return any(self._match(r, key) for r in self.rows)
    def get(self, **key):
        return [r for r in self.rows if self._match(r, key)]
    def add(self, row):
        self.rows.append(row)
        save_json(self.rows, self.path)

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def mean_sd(v):
    v = np.asarray(v, dtype=float)
    return (float(v.mean()), float(v.std(ddof=1)) if len(v) > 1 else 0.0) if len(v) else (float("nan"), float("nan"))

def fmt_ms(v, scale=1.0, nd=3):
    m, s = mean_sd(np.asarray(v, dtype=float) * scale)
    return f"{m:.{nd}f} ± {s:.{nd}f}"

In [ ]:
# 1.3b  image paths, cache and datasets ----------------------------------------------------------
# A "sample" is (relative_path, label). Relative paths are relative to DATA_ROOT, so the same CSV
# split files work on your PC and on Colab. Absolute paths are allowed too (GAN images).

def to_rel(p):
    return os.path.relpath(p, DATA_ROOT).replace("\\", "/")

def abs_data(r):
    return r if os.path.isabs(r) else os.path.join(DATA_ROOT, *r.split("/"))

def cache_path(r):
    return os.path.join(CACHE_ROOT, *r.split("/"))

def resolve(r):
    if os.path.isabs(r):
        return r
    c = cache_path(r)
    return c if os.path.exists(c) else abs_data(r)

def load_rgb(path, draft=512):
    img = Image.open(path)
    if draft and img.format == "JPEG":
        img.draft("RGB", (draft, draft))   # decodes large JPEGs at reduced size = much faster
    return img.convert("RGB")

def build_cache(rel_paths, max_side=CACHE_MAX_SIDE):
    # Shrink large field photos once. Small images (PlantVillage 256x256) are never cached.
    made = 0
    for r in tqdm(rel_paths, desc="cache"):
        dst = cache_path(r)
        if os.path.exists(dst):
            continue
        img = load_rgb(abs_data(r), draft=max_side * 2)
        if max(img.size) > max_side:
            img.thumbnail((max_side, max_side), Image.BICUBIC)
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        img.save(dst + ".tmp.jpg", "JPEG", quality=95)
        os.replace(dst + ".tmp.jpg", dst)
        made += 1
    return made

def list_folder(root, class_names=CLASSES, class_to_idx=None):
    class_to_idx = class_to_idx or {c: i for i, c in enumerate(CLASSES)}
    out = []
    for c in class_names:
        d = os.path.join(root, c)
        if not os.path.isdir(d):
            continue
        for fn in sorted(os.listdir(d)):
            if fn.lower().endswith((".jpg", ".jpeg", ".png")):
                out.append((to_rel(os.path.join(d, fn)), class_to_idx[c]))
    return out

def dev_cap(samples, n=DEV_PER_CLASS, seed=0):
    # FAST_DEV_RUN: keep only n images per class
    if not FAST_DEV_RUN:
        return list(samples)
    rng = random.Random(seed)
    by = defaultdict(list)
    for s in samples:
        by[s[1]].append(s)
    out = []
    for k in sorted(by):
        v = by[k][:]
        rng.shuffle(v)
        out += v[:n]
    return out

def write_split_csv(samples, path, extra_cols=None):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path + ".tmp", "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["relpath", "label", "class"] + (extra_cols or []))
        for s in samples:
            w.writerow([s[0], s[1], CLASSES[s[1]]] + list(s[2:]))
    os.replace(path + ".tmp", path)

def read_split_csv(path):
    with open(path, encoding="utf-8") as f:
        rows = list(csv.reader(f))
    return [(r[0], int(r[1])) + tuple(r[3:]) for r in rows[1:]]

IMAGENET_MEAN, IMAGENET_STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

def train_transforms(size, groups=("geometric", "photometric", "occlusion")):
    # Section 2.3 of the paper. `groups` is only changed by the augmentation ablation (Section 12).
    t = []
    if "geometric" in groups:
        t += [T.RandomResizedCrop(size, scale=(0.6, 1.0)), T.RandomHorizontalFlip(), T.RandomVerticalFlip(),
              T.RandomRotation(35)]
    else:
        t += [T.Resize((size, size))]
    if "photometric" in groups:
        t += [T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.05)]
    t += [T.ToTensor(), T.Normalize(IMAGENET_MEAN, IMAGENET_STD)]
    if "occlusion" in groups:
        t += [T.RandomErasing(p=0.2, scale=(0.02, 0.1))]   # stands in for "coarse dropout"
    return T.Compose(t)

def eval_transforms(size):
    return T.Compose([T.Resize((size, size)), T.ToTensor(), T.Normalize(IMAGENET_MEAN, IMAGENET_STD)])

class SampleDataset(Dataset):
    def __init__(self, samples, transform):
        self.samples = [(s[0], int(s[1])) for s in samples]
        self.transform = transform
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, i):
        r, y = self.samples[i]
        return self.transform(load_rgb(resolve(r))), y, r

def make_loader(samples, transform, batch_size, shuffle=False, seed=0, drop_last=False):
    g = torch.Generator()
    g.manual_seed(seed)
    return DataLoader(SampleDataset(samples, transform), batch_size=batch_size, shuffle=shuffle,
                      generator=g if shuffle else None, num_workers=NUM_WORKERS,
                      pin_memory=(DEVICE.type == "cuda"), drop_last=drop_last)

def class_weights_for(samples, n=len(CLASSES)):
    counts = np.bincount([int(s[1]) for s in samples], minlength=n).astype(np.float64)
    counts[counts == 0] = 1.0
    return torch.tensor(counts.sum() / (n * counts), dtype=torch.float32)

In [ ]:
# 1.3c  models: EXACT layout of the saved paper checkpoints ----------------------------------------
# The checkpoints were saved as nn.Sequential(body, [pool], head). Rebuilding that layout lets all five
# load with strict=True (verified key by key). index 0 = body, last index = head.
class GAPFlatten(nn.Module):
    def forward(self, x):
        return torch.flatten(F.adaptive_avg_pool2d(x, 1), 1)

INPUT_SIZE = {"resnet50": 224, "mobilenet_v2": 224, "densenet121": 224, "vgg16": 224, "inception_v3": 299}
FEAT_SPLIT = {"resnet50": 1, "mobilenet_v2": 1, "densenet121": 1, "vgg16": 3, "inception_v3": 1}  # head[:k] -> embedding
FEAT_DIM   = {"resnet50": 2048, "mobilenet_v2": 1280, "densenet121": 1024, "vgg16": 512, "inception_v3": 2048}
DETECTED = load_json(os.path.join(RESULTS_ROOT, "detected_settings.json"), {"densenet_relu": True})

def _torchvision_model(name, pretrained):
    weights = {"resnet50": "ResNet50_Weights.IMAGENET1K_V2", "mobilenet_v2": "MobileNet_V2_Weights.IMAGENET1K_V1",
               "densenet121": "DenseNet121_Weights.IMAGENET1K_V1", "vgg16": "VGG16_Weights.IMAGENET1K_V1",
               "inception_v3": "Inception_V3_Weights.IMAGENET1K_V1"}[name]
    fn = getattr(tv.models, name)
    kw = {"aux_logits": True, "init_weights": False} if name == "inception_v3" else {}
    try:
        w = None
        if pretrained:
            enum_name, member = weights.split(".")
            w = getattr(getattr(tv.models, enum_name), member)
        return fn(weights=w, **kw)
    except (AttributeError, TypeError):   # very old torchvision
        return fn(pretrained=pretrained, **kw)

def build_model(name, pretrained=True, num_classes=4):
    m = _torchvision_model(name, pretrained)
    if name == "resnet50":
        mods = [nn.Sequential(*list(m.children())[:-2]),
                nn.Sequential(GAPFlatten(), nn.Linear(2048, num_classes))]
    elif name == "densenet121":
        head = nn.Sequential(nn.Flatten(), nn.Linear(1024, 256), nn.ReLU(True), nn.Dropout(0.2), nn.Linear(256, num_classes))
        mods = [m.features, nn.ReLU(True) if DETECTED.get("densenet_relu", True) else nn.Identity(),
                nn.AdaptiveAvgPool2d(1), head]
    elif name == "mobilenet_v2":
        head = nn.Sequential(nn.Flatten(), nn.Linear(1280, 128), nn.ReLU(True), nn.Dropout(0.2), nn.Linear(128, num_classes))
        mods = [m.features, nn.AdaptiveAvgPool2d(1), head]
    elif name == "vgg16":
        head = nn.Sequential(nn.Flatten(), nn.Linear(512 * 7 * 7, 512), nn.ReLU(True), nn.Dropout(0.5), nn.Linear(512, num_classes))
        mods = [m.features, nn.AdaptiveAvgPool2d(7), head]
    elif name == "inception_v3":
        skip = {"AuxLogits", "avgpool", "dropout", "fc"}
        body = nn.Sequential(*[c for n, c in m.named_children() if n not in skip])
        head = nn.Sequential(nn.Flatten(), nn.Linear(2048, 256), nn.ReLU(True), nn.Dropout(0.3), nn.Linear(256, num_classes))
        mods = [body, nn.AdaptiveAvgPool2d(1), head]
    else:
        raise ValueError(name)
    model = nn.Sequential(*mods)
    model.arch = name
    return model

def body_of(model):
    return model[0]

def embed(model, x):
    # pooled embedding (t-SNE, CORAL, DANN, similarity checks)
    h = model[-1]
    for mod in list(model)[:-1]:
        x = mod(x)
    for mod in list(h)[:FEAT_SPLIT[model.arch]]:
        x = mod(x)
    return x

def logits_from_embedding(model, f):
    for mod in list(model[-1])[FEAT_SPLIT[model.arch]:]:
        f = mod(f)
    return f

def load_full_checkpoint(model, path):
    sd = torch_load(path)
    if isinstance(sd, dict) and "state_dict" in sd:
        sd = sd["state_dict"]
    model.load_state_dict(sd, strict=True)
    return model

def transfer_body(model, path):
    # Copy ONLY the body (keys "0.*"); the head stays freshly initialised (paper protocol).
    sd = torch_load(path)
    if isinstance(sd, dict) and "state_dict" in sd:
        sd = sd["state_dict"]
    own = model.state_dict()
    body_keys = [k for k in own if k.startswith("0.")]
    matched = {k: sd[k] for k in body_keys if k in sd and tuple(sd[k].shape) == tuple(own[k].shape)}
    assert len(matched) == len(body_keys), f"only {len(matched)}/{len(body_keys)} body tensors matched in {path}"
    model.load_state_dict(matched, strict=False)
    return len(matched)

def paper_classifier(name):
    m = build_model(name, pretrained=False)
    load_full_checkpoint(m, os.path.join(CKPT_ROOT, CKPT_FILES[name]))
    return m.to(DEVICE).eval()

In [ ]:
# 1.3d  resumable two-phase training (Section 2.4 of the paper) ------------------------------------
# Phase 1: body frozen, head trained 5 epochs @1e-3. Phase 2: everything trained up to 25 epochs @1e-4,
# early stopping (patience 8) on validation accuracy, best weights restored. AdamW, wd 1e-4,
# class-weighted cross-entropy, effective batch 16.
# NEW: after every epoch the model + optimizer + counters are saved to <run_dir>/last.pt, so an
# interrupted run continues from the last finished epoch when the cell is run again.

def set_body_trainable(model, flag):
    for p in body_of(model).parameters():
        p.requires_grad = flag

def run_epoch(model, loader, criterion, optimizer=None, scaler=None):
    train = optimizer is not None
    model.train(train)
    tot, correct, loss_sum = 0, 0, 0.0
    if train:
        optimizer.zero_grad(set_to_none=True)
    n_batches = len(loader)
    for i, (x, y, _) in enumerate(loader):
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
        with torch.set_grad_enabled(train), autocast_ctx():
            logits = model(x)
            loss = criterion(logits, y)
        if train:
            scaler.scale(loss / ACCUM_STEPS).backward()
            if (i + 1) % ACCUM_STEPS == 0 or (i + 1) == n_batches:
                scaler.step(optimizer); scaler.update()
                optimizer.zero_grad(set_to_none=True)
        loss_sum += float(loss.item()) * x.size(0)
        correct += int((logits.argmax(1) == y).sum().item())
        tot += x.size(0)
    return correct / max(tot, 1), loss_sum / max(tot, 1)

def fit_two_phase(model, train_samples, val_samples, run_dir, seed, train_tf=None, cfg=None, tag=""):
    cfg = dict(TRAIN_CFG, **(cfg or {}))
    size = INPUT_SIZE[model.arch]
    train_tf = train_tf or train_transforms(size)
    os.makedirs(run_dir, exist_ok=True)
    last_p, best_p, done_p = [os.path.join(run_dir, f) for f in ("last.pt", "best.pt", "done.json")]
    model.to(DEVICE)
    if os.path.exists(done_p) and os.path.exists(best_p):          # finished earlier
        model.load_state_dict(torch_load(best_p))
        return model, load_json(done_p)

    criterion = nn.CrossEntropyLoss(weight=class_weights_for(train_samples).to(DEVICE))
    val_loader = make_loader(val_samples, eval_transforms(size), EVAL_BATCH)
    p1, p2 = cfg["phase1_epochs"], cfg["phase2_epochs"]
    st = dict(epoch=0, best_acc=-1.0, best_epoch=-1, bad=0, history=[])
    ckpt = torch_load(last_p) if os.path.exists(last_p) else None
    if ckpt is not None:
        model.load_state_dict(ckpt["model"])
        st = ckpt["state"]
        log(f"  {tag} resuming from epoch {st['epoch']}")
    scaler = make_scaler()
    if ckpt is not None and ckpt.get("scaler"):
        scaler.load_state_dict(ckpt["scaler"])
    optimizer, cur_phase = None, None
    for ep in range(st["epoch"], p1 + p2):
        phase = 1 if ep < p1 else 2
        if phase != cur_phase:
            set_body_trainable(model, phase == 2)
            params = [p for p in model.parameters() if p.requires_grad]
            optimizer = torch.optim.AdamW(params, lr=cfg["phase1_lr"] if phase == 1 else cfg["phase2_lr"],
                                          weight_decay=cfg["weight_decay"])
            if ckpt is not None and ckpt.get("phase") == phase and ckpt.get("opt") is not None:
                optimizer.load_state_dict(ckpt["opt"])
            ckpt, cur_phase = None, phase
        t0 = time.time()
        # the shuffle order depends only on (seed, epoch) -> identical after a resume
        train_loader = make_loader(train_samples, train_tf, MICRO_BATCH, shuffle=True, seed=seed * 1000 + ep)
        tr_acc, tr_loss = run_epoch(model, train_loader, criterion, optimizer, scaler)
        va_acc, va_loss = run_epoch(model, val_loader, criterion)
        stop = False
        if phase == 2:
            if va_acc > st["best_acc"]:
                st.update(best_acc=va_acc, best_epoch=ep + 1, bad=0)
                torch_save_atomic(model.state_dict(), best_p)
            else:
                st["bad"] += 1
                stop = st["bad"] >= cfg["patience"]
        st["epoch"] = ep + 1
        st["history"].append(dict(epoch=ep + 1, phase=phase, train_acc=tr_acc, train_loss=tr_loss,
                                   val_acc=va_acc, val_loss=va_loss, sec=round(time.time() - t0, 1)))
        torch_save_atomic(dict(model=model.state_dict(), opt=optimizer.state_dict(), scaler=scaler.state_dict(),
                               phase=phase, state=st), last_p)
        print(f"  {tag} ep {ep+1:2d}/{p1+p2} [phase {phase}] train_acc={tr_acc:.3f} val_acc={va_acc:.3f} "
              f"({time.time()-t0:.0f}s)")
        if stop:
            print(f"  {tag} early stopping (best val_acc={st['best_acc']:.4f} @ epoch {st['best_epoch']})")
            break
    if not os.path.exists(best_p):          # e.g. phase2_epochs == 0
        torch_save_atomic(model.state_dict(), best_p)
    model.load_state_dict(torch_load(best_p))
    info = dict(best_val_acc=st["best_acc"], best_epoch=st["best_epoch"], epochs_run=st["epoch"],
                history=st["history"])
    save_json(info, done_p)
    if os.path.exists(last_p):
        os.remove(last_p)                   # the optimizer state is no longer needed
    return model, info

def cleanup_run(run_dir, keep_weights_as=None):
    # after the metrics are stored: optionally copy the weights somewhere, then free disk space
    best_p = os.path.join(run_dir, "best.pt")
    if keep_weights_as and os.path.exists(best_p):
        os.makedirs(os.path.dirname(keep_weights_as), exist_ok=True)
        shutil.copyfile(best_p, keep_weights_as)
    for f in ("best.pt", "last.pt"):
        p = os.path.join(run_dir, f)
        if os.path.exists(p):
            os.remove(p)

In [ ]:
# 1.3e  evaluation, metrics, bootstrap ------------------------------------------------------------
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support, f1_score, confusion_matrix,
                             roc_auc_score)

@torch.no_grad()
def predict(model, samples, size=None, batch_size=None):
    size = size or INPUT_SIZE[model.arch]
    model.eval().to(DEVICE)
    loader = make_loader(samples, eval_transforms(size), batch_size or EVAL_BATCH)
    probs, labels, paths = [], [], []
    for x, y, p in loader:
        with autocast_ctx():
            out = model(x.to(DEVICE))
        probs.append(torch.softmax(out.float(), 1).cpu().numpy())
        labels.append(y.numpy()); paths += list(p)
    probs = np.concatenate(probs) if probs else np.zeros((0, len(CLASSES)))
    return dict(probs=probs, labels=np.concatenate(labels) if labels else np.zeros(0, int), paths=paths)

def metrics_from(labels, probs, present=None):
    # present = class indices that exist in this dataset (PlantDoc has no black rot).
    # preds_4way: argmax over all 4 outputs (a black-rot prediction on PlantDoc counts as an error)
    # preds_restricted: argmax over the present classes only
    present = sorted(set(labels.tolist())) if present is None else present
    preds = probs.argmax(1)
    restricted = np.array(present)[probs[:, present].argmax(1)] if len(probs) else preds
    out = {}
    for name, pr in [("", preds), ("restricted_", restricted)]:
        p, r, f1, _ = precision_recall_fscore_support(labels, pr, labels=present, average="macro", zero_division=0)
        out[name + "accuracy"] = float(accuracy_score(labels, pr))
        out[name + "precision_macro"] = float(p)
        out[name + "recall_macro"] = float(r)
        out[name + "f1_macro"] = float(f1)
    pc = precision_recall_fscore_support(labels, preds, labels=list(range(len(CLASSES))), zero_division=0)
    out["per_class"] = {CLASSES[i]: dict(precision=float(pc[0][i]), recall=float(pc[1][i]), f1=float(pc[2][i]),
                                         support=int(pc[3][i])) for i in range(len(CLASSES))}
    out["confusion"] = confusion_matrix(labels, preds, labels=list(range(len(CLASSES)))).tolist()
    try:
        if len(present) == len(CLASSES):
            out["roc_auc_ovr"] = float(roc_auc_score(labels, probs, multi_class="ovr"))
    except ValueError:
        pass
    out["n"] = int(len(labels))
    return out

def evaluate(model, samples, present=None):
    pr = predict(model, samples)
    m = metrics_from(pr["labels"], pr["probs"], present)
    return m, pr

def bootstrap_ci(labels, preds, fn, n=None, seed=0, alpha=0.05):
    n = n or BOOTSTRAP_N
    rng = np.random.RandomState(seed)
    idx = np.arange(len(labels))
    vals = []
    for _ in range(n):
        b = rng.choice(idx, len(idx), replace=True)
        vals.append(fn(labels[b], preds[b]))
    lo, hi = np.percentile(vals, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    return float(fn(labels, preds)), float(lo), float(hi)

def macro_f1_on(present):
    return lambda y, p: f1_score(y, p, labels=present, average="macro", zero_division=0)

def save_fig(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(os.path.join(FIG_DIR, f"{name}.{ext}"), dpi=300, bbox_inches="tight")
    print("saved figure:", os.path.join(FIG_DIR, name + ".png"))

def md_table(header, rows):
    s = "| " + " | ".join(header) + " |\n|" + "|".join(["---"] * len(header)) + "|\n"
    for r in rows:
        s += "| " + " | ".join(str(x) for x in r) + " |\n"
    return s

SUMMARY_PATH = os.path.join(RESULTS_ROOT, "section_summaries.json")
def put_summary(section, text):
    d = load_json(SUMMARY_PATH, {})
    d[section] = text
    save_json(d, SUMMARY_PATH)
    print(text)

print("library ready")

## 2. Data preparation (run once; the lists are saved as CSV and reused)

* PlantVillage leakage-free split: `train / val / test` (2,271 / 486 / 490).
* PlantVillage **raw leaky** folder (9,819 files) with the physical-leaf id of every file.
* FGVC8: a fixed, de-duplicated random sample of **700 images per class** (zero-shot evaluation, like the paper's 2,800). Its first **500 per class** form the data-efficiency / domain-adaptation pool, and a fixed 20% of that (400 images) is the field test split.
* PlantDoc: 287 images, 3 classes.
* Large field photos are shrunk once into `data_cache/` (max side 384 px). This only affects speed; every model still sees 224×224 (299 for InceptionV3).

> Keep the CSV files in `result_new/splits/`. If you move to Colab, copy that folder so that exactly the same images are used.

In [ ]:
# 2.1  file lists -------------------------------------------------------------------------------
from sklearn.model_selection import train_test_split

PV = {sp: dev_cap(list_folder(os.path.join(PV_ROOT, sp))) for sp in ("train", "val", "test")}
PLANTDOC = dev_cap(list_folder(PLANTDOC_ROOT))
PLANTDOC_PRESENT = sorted(set(s[1] for s in PLANTDOC))          # [0, 2, 3]: no black rot
ALL4 = list(range(len(CLASSES)))

# physical-leaf id: strip every on-disk augmentation suffix used in the raw PlantVillage folder
# (_90deg/_180deg/_270deg, _flipLR/_flipTB, _new30degFlipLR/_new30degFlipTB, _newPixel25, _newGRR)
AUG_SUFFIX_RE = re.compile(r"_(\d+deg|new\d+degFlip(LR|TB)|flip(LR|TB)|newPixel\d+|newGRR)$", re.IGNORECASE)
def base_leaf_id(rel):
    stem = os.path.splitext(os.path.basename(rel))[0]
    prev = None
    while prev != stem:
        prev, stem = stem, AUG_SUFFIX_RE.sub("", stem)
    return rel.split("/")[-2] + "/" + stem

LEAKY_ALL = list_folder(LEAKY_ROOT)
if FAST_DEV_RUN:
    LEAKY_ALL = dev_cap(LEAKY_ALL, n=16)
LEAKY_BASE = [base_leaf_id(r) for r, _ in LEAKY_ALL]

def build_fgvc8_pools():
    p_eval = os.path.join(SPLITS_DIR, f"fgvc8_eval_{FGVC8_POOL_EVAL}_per_class.csv")
    p_de   = os.path.join(SPLITS_DIR, f"fgvc8_pool_{FGVC8_POOL_DE}_per_class.csv")
    if os.path.exists(p_eval) and os.path.exists(p_de):
        return read_split_csv(p_eval), read_split_csv(p_de)
    rng = random.Random(2026)
    ev, de = [], []
    for ci, c in enumerate(CLASSES):
        files = sorted(os.listdir(os.path.join(FGVC8_ROOT, c)))
        rng.shuffle(files)
        seen, chosen = set(), []
        for fn in tqdm(files, desc=f"FGVC8 {c}", leave=False):
            full = os.path.join(FGVC8_ROOT, c, fn)
            with open(full, "rb") as fh:
                h = hashlib.md5(fh.read()).hexdigest()
            if h in seen:
                continue                       # exact duplicate file -> skip
            seen.add(h)
            chosen.append(to_rel(full))
            if len(chosen) == FGVC8_POOL_EVAL:
                break
        ev += [(r, ci) for r in chosen]
        de += [(r, ci) for r in chosen[:FGVC8_POOL_DE]]
    write_split_csv(ev, p_eval); write_split_csv(de, p_de)
    return ev, de

FGVC8_EVAL, FGVC8_DE = build_fgvc8_pools()

# fixed field test split (same for data-efficiency AND domain adaptation)
_de_labels = np.array([s[1] for s in FGVC8_DE])
_pool_idx, _test_idx = train_test_split(np.arange(len(FGVC8_DE)), test_size=FIELD_TEST_FRAC,
                                        random_state=FIELD_SPLIT_SEED, stratify=_de_labels)
FIELD_TRAIN_POOL = [FGVC8_DE[i] for i in _pool_idx]
FIELD_TEST       = [FGVC8_DE[i] for i in _test_idx]
write_split_csv(FIELD_TRAIN_POOL, os.path.join(SPLITS_DIR, "fgvc8_field_train_pool.csv"))
write_split_csv(FIELD_TEST, os.path.join(SPLITS_DIR, "fgvc8_field_test.csv"))
for sp in PV:
    write_split_csv(PV[sp], os.path.join(SPLITS_DIR, f"plantvillage_leakfree_{sp}.csv"))
write_split_csv(PLANTDOC, os.path.join(SPLITS_DIR, "plantdoc_external.csv"))

print(f"PlantVillage leak-free: " + ", ".join(f"{k}={len(v)}" for k, v in PV.items()))
print(f"PlantVillage raw leaky: {len(LEAKY_ALL)} files from {len(set(LEAKY_BASE))} physical leaves")
print(f"FGVC8 eval pool: {len(FGVC8_EVAL)} | DE pool: {len(FGVC8_DE)} -> train pool {len(FIELD_TRAIN_POOL)}, "
      f"field test {len(FIELD_TEST)}")
print(f"PlantDoc: {len(PLANTDOC)} images, classes present = {[CLASSES[i] for i in PLANTDOC_PRESENT]}")

In [ ]:
# 2.2  shrink the big field photos once + check that every listed file exists ------------------------
if RUN["data_prep"]:
    to_cache = sorted(set([s[0] for s in FGVC8_EVAL] + [s[0] for s in PLANTDOC]))
    n = build_cache(to_cache)
    print(f"cache: {n} new images written ({len(to_cache)} total listed)")
missing = [s[0] for s in PV["train"] + PV["val"] + PV["test"] + FGVC8_EVAL + PLANTDOC + LEAKY_ALL
           if not os.path.exists(resolve(s[0]))]
print("missing files:", len(missing), missing[:5])
assert not missing, "some listed files are missing - check DATA_ROOT"

In [ ]:
# 2.3  import results that were already produced earlier (YOLO comparison, GAN generator, old t-SNE numbers)
IMPORT_DIR = os.path.join(RESULTS_ROOT, "imported_old_runs")
os.makedirs(IMPORT_DIR, exist_ok=True)
GAN_DIR = os.path.join(RESULTS_ROOT, "gan")
os.makedirs(GAN_DIR, exist_ok=True)
wanted = {
    "yolo_compare_out/yolo_comparison_raw.json": os.path.join(IMPORT_DIR, "yolo_comparison_raw_30ep.json"),
    "tsne_out/quantitative_summary.txt": os.path.join(IMPORT_DIR, "old_tsne_quantitative_summary.txt"),
    "leaky_split_out/leaky_split_results.json": os.path.join(IMPORT_DIR, "old_leaky_split_results.json"),
    "gan_out/generator.pt": os.path.join(GAN_DIR, "generator_imported.pt"),
}
for zp in OLD_RESULT_ZIPS:
    if not os.path.exists(zp):
        continue
    with zipfile.ZipFile(zp) as zf:
        names = set(zf.namelist())
        for src, dst in wanted.items():
            if src in names and not os.path.exists(dst):
                with zf.open(src) as fi, open(dst, "wb") as fo:
                    shutil.copyfileobj(fi, fo)
                print("imported", src, "from", os.path.basename(zp))
print("imported files:", sorted(os.listdir(IMPORT_DIR)), "| GAN dir:", sorted(os.listdir(GAN_DIR)))

### 2.4 Speed test: how long will the training sections take on this machine?

A few forward+backward passes on random images give a time per image. The estimate is rough (image loading is not included), but it shows quickly whether a section is practical here or should go to Colab.

In [ ]:
# 2.4  speed benchmark ---------------------------------------------------------------------------
def sec_per_train_image(arch, iters=3):
    m = build_model(arch, pretrained=False).to(DEVICE).train()
    opt = torch.optim.AdamW(m.parameters(), lr=1e-4)
    x = torch.randn(MICRO_BATCH, 3, INPUT_SIZE[arch], INPUT_SIZE[arch], device=DEVICE)
    y = torch.randint(0, 4, (MICRO_BATCH,), device=DEVICE)
    scaler = make_scaler()
    times = []
    for i in range(iters + 1):
        t0 = time.time()
        with autocast_ctx():
            loss = F.cross_entropy(m(x), y)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); opt.zero_grad()
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        if i > 0:
            times.append(time.time() - t0)
    del m, opt
    return float(np.mean(times)) / MICRO_BATCH

if RUN["data_prep"]:
    SPEED = {}
    for a in sorted(set(DE_BACKBONES + ["resnet50"])):
        v = oom_guard(f"speed test {a}", sec_per_train_image, a)
        SPEED[a] = v if v is not None else float("nan")
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()
    EPOCHS_TYPICAL = 20      # 5 frozen + ~15 fine-tuning epochs before early stopping (typical)
    def hours(n_images, arch="resnet50", epochs=EPOCHS_TYPICAL, runs=1):
        return SPEED[arch] * n_images * 1.25 * epochs * runs / 3600   # 1.25: validation + loading overhead

    n_pool = len(FIELD_TRAIN_POOL)
    de_imgs = sum(0.85 * f * n_pool for f in DE_FRACTIONS)
    est = {}
    for a in DE_BACKBONES:
        n_init = 3 if (a == "resnet50" and DE_INCLUDE_LEAKY_INIT) else 2
        est[f"8. data-efficiency {a}"] = hours(de_imgs, a, runs=n_init * len(SEEDS))
    leaky_train = int(0.7 * len(LEAKY_ALL))
    est["9. leaky split (3 seeds) + honest retrain (3 seeds)"] = hours(leaky_train, runs=len(SEEDS)) + hours(len(PV["train"]), runs=len(SEEDS))
    est["10. domain adaptation (3 methods x 3 seeds)"] = hours(2 * len(PV["train"]), epochs=DA_EPOCHS, runs=len(DA_METHODS) * len(SEEDS))
    est["11. GAN retrain (2 conditions x 3 seeds)"] = hours(len(PV["train"]) + GAN_N_GENERATE, runs=2 * len(SEEDS))
    print(f"seconds per training image: " + ", ".join(f"{k}={v:.4f}" for k, v in SPEED.items()))
    for k, v in est.items():
        print(f"  {k:55s} ~ {v:6.1f} h")
    print("If a number is far beyond what you can leave the PC running, run that section on Colab (RUN_ENV='colab').")

## 3. Sanity check + field error analysis (inference only)

**Why:** before new numbers go into the paper, the paper checkpoints must reproduce the paper's numbers with this code. The same predictions then answer:

* **Reviewer A, W1** and **Reviewer B (Results)**: why does zero-shot fail? We add confusion matrices, the most confident errors, and accuracy against brightness / blur / vegetation fraction.
* **Reviewer B**: bootstrap 95% confidence intervals for the field results.
* **Reviewer A, W5**: zero-shot accuracy of **all five** backbones, plus the latency of each model on this machine (batch 1, CPU or GPU).

The FGVC8 evaluation sample is a new fixed random sample, so its zero-shot number can differ slightly from the paper's 32.86%.

In [ ]:
# 3.1  DenseNet121: was the checkpoint saved with a ReLU between features and pooling? (test both) ------
if RUN["sanity"] and "detected" not in DETECTED:
    res = {}
    for flag in (True, False):
        DETECTED["densenet_relu"] = flag
        m = paper_classifier("densenet121")
        pr = predict(m, PV["val"])
        nll = float(-np.log(pr["probs"][np.arange(len(pr["labels"])), pr["labels"]] + 1e-9).mean())
        res[flag] = (float((pr["probs"].argmax(1) == pr["labels"]).mean()), -nll)
        print(f"densenet relu={flag}: val acc={res[flag][0]:.4f}  NLL={-res[flag][1]:.4f}")
    DETECTED["densenet_relu"] = bool(max(res, key=lambda k: res[k]))
    DETECTED["detected"] = True
    save_json(DETECTED, os.path.join(RESULTS_ROOT, "detected_settings.json"))
print("DenseNet121 ReLU after features:", DETECTED["densenet_relu"])

In [ ]:
# 3.2  paper checkpoints on PlantVillage test + zero-shot on PlantDoc and FGVC8 (all five backbones) ---
PAPER = {"pv_acc": {"resnet50": 99.86, "vgg16": 99.86, "densenet121": 99.73, "inception_v3": 99.73, "mobilenet_v2": 99.12},
         "plantdoc_zs": 41.46, "fgvc8_zs": 32.86}
SANITY_DIR = os.path.join(RESULTS_ROOT, "sanity")
os.makedirs(SANITY_DIR, exist_ok=True)
sanity = ResultStore(os.path.join(SANITY_DIR, "results.json"))
ALL_BACKBONES = ["resnet50", "vgg16", "densenet121", "inception_v3", "mobilenet_v2"]
DATASETS = {"pv_test": (lambda: PV["test"], ALL4), "plantdoc": (lambda: PLANTDOC, PLANTDOC_PRESENT),
            "fgvc8": (lambda: FGVC8_EVAL, ALL4)}

def save_predictions(pr, path):
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["relpath", "label", "pred", "conf"] + [f"p_{c}" for c in CLASSES])
        for r, y, p in zip(pr["paths"], pr["labels"], pr["probs"]):
            w.writerow([r, int(y), int(p.argmax()), f"{p.max():.4f}"] + [f"{v:.4f}" for v in p])

PRED = {}
if RUN["sanity"]:
    for bb in ALL_BACKBONES:
        model = None
        for ds, (get, present) in DATASETS.items():
            pred_csv = os.path.join(SANITY_DIR, f"pred_{bb}_{ds}.csv")
            if sanity.has(backbone=bb, dataset=ds) and os.path.exists(pred_csv):
                continue
            if model is None:
                model = paper_classifier(bb)
            m, pr = evaluate(model, get(), present)
            save_predictions(pr, pred_csv)
            sanity.add(dict(backbone=bb, dataset=ds, **{k: v for k, v in m.items()}))
            print(f"{bb:13s} {ds:9s} acc={m['accuracy']*100:6.2f}%  restricted={m['restricted_accuracy']*100:6.2f}%  "
                  f"macroF1={m['f1_macro']:.3f}")
        del model

    rows = []
    for bb in ALL_BACKBONES:
        g = {r["dataset"]: r for r in sanity.get(backbone=bb)}
        rows.append([bb, f"{g['pv_test']['accuracy']*100:.2f}", PAPER["pv_acc"][bb],
                     f"{g['plantdoc']['accuracy']*100:.2f}", f"{g['plantdoc']['restricted_accuracy']*100:.2f}",
                     f"{g['fgvc8']['accuracy']*100:.2f}", f"{g['fgvc8']['f1_macro']:.3f}"])
    txt = ("Paper checkpoints re-evaluated (paper: PlantDoc zero-shot 41.46%, FGVC8 32.86% for ResNet50).\n"
           "'PD 3-way' = argmax restricted to the three PlantDoc classes.\n\n" +
           md_table(["backbone", "PV test acc %", "paper %", "PlantDoc ZS %", "PD 3-way %", "FGVC8 ZS %", "FGVC8 macro-F1"], rows))
    put_summary("3.2 sanity + zero-shot all backbones", txt)
    r = {x["dataset"]: x for x in sanity.get(backbone="resnet50")}
    for key, ds in [("plantdoc_zs", "plantdoc"), ("fgvc8_zs", "fgvc8")]:
        for variant in ("accuracy", "restricted_accuracy"):
            if abs(r[ds][variant] * 100 - PAPER[key]) < 1.0:
                print(f"-> ResNet50 {ds} {variant} reproduces the paper ({r[ds][variant]*100:.2f}% vs {PAPER[key]}%)")

In [ ]:
# 3.3  field error analysis for ResNet50: bootstrap CIs, confusion matrices, most-confident errors -----
def read_pred_csv(path):
    with open(path, encoding="utf-8") as f:
        rows = list(csv.DictReader(f))
    probs = np.array([[float(r[f"p_{c}"]) for c in CLASSES] for r in rows])
    return dict(paths=[r["relpath"] for r in rows], labels=np.array([int(r["label"]) for r in rows]), probs=probs)

if RUN["sanity"]:
    ci_rows = []
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), constrained_layout=True)
    for ax, (ds, title, present) in zip(axes, [("plantdoc", "PlantDoc (zero-shot)", PLANTDOC_PRESENT),
                                              ("fgvc8", "FGVC8 (zero-shot)", ALL4)]):
        pr = read_pred_csv(os.path.join(SANITY_DIR, f"pred_resnet50_{ds}.csv"))
        PRED[ds] = pr
        y, p = pr["labels"], pr["probs"].argmax(1)
        acc = bootstrap_ci(y, p, lambda a, b: float((a == b).mean()))
        f1 = bootstrap_ci(y, p, macro_f1_on(present))
        ci_rows.append([ds, len(y), f"{acc[0]*100:.1f} [{acc[1]*100:.1f}, {acc[2]*100:.1f}]",
                        f"{f1[0]:.3f} [{f1[1]:.3f}, {f1[2]:.3f}]"])
        cm = confusion_matrix(y, p, labels=ALL4)
        rows_keep = present
        cmn = cm[rows_keep] / np.maximum(cm[rows_keep].sum(1, keepdims=True), 1)
        im = ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
        for i in range(cmn.shape[0]):
            for j in range(cmn.shape[1]):
                ax.text(j, i, f"{cm[rows_keep][i, j]}", ha="center", va="center",
                        color="white" if cmn[i, j] > 0.6 else "black", fontsize=9)
        ax.set_xticks(range(4)); ax.set_xticklabels(CLASS_TITLES, rotation=30, ha="right")
        ax.set_yticks(range(len(rows_keep))); ax.set_yticklabels([CLASS_TITLES[i] for i in rows_keep])
        ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(title)
    fig.colorbar(im, ax=axes, shrink=0.8, label="row-normalised")
    save_fig(fig, "fig_zero_shot_confusion_resnet50"); plt.show()
    # the FGVC8 cross-validation confusion (Table 7 / Figure 5 of the paper) is an ADAPTATION result; this one is zero-shot
    put_summary("3.3 bootstrap 95% CI (ResNet50 zero-shot)",
                md_table(["dataset", "n", "accuracy % [95% CI]", "macro-F1 [95% CI]"], ci_rows))

    # most confident errors (what does the model get wrong with high confidence?)
    for ds in ("plantdoc", "fgvc8"):
        pr = PRED[ds]
        pred, conf = pr["probs"].argmax(1), pr["probs"].max(1)
        wrong = np.where(pred != pr["labels"])[0]
        wrong = wrong[np.argsort(-conf[wrong])][:16]
        if len(wrong) == 0:
            continue
        fig, axes = plt.subplots(4, 4, figsize=(11, 11))
        for ax in axes.ravel():
            ax.axis("off")
        for ax, i in zip(axes.ravel(), wrong):
            ax.imshow(load_rgb(resolve(pr["paths"][i]), draft=384))
            ax.set_title(f"true: {CLASS_TITLES[pr['labels'][i]]}\npred: {CLASS_TITLES[pred[i]]} ({conf[i]:.2f})", fontsize=8)
        fig.suptitle(f"{ds}: 16 most confident zero-shot errors (ResNet50, leakage-free)")
        save_fig(fig, f"fig_most_confident_errors_{ds}"); plt.show()

        # what the errors go to
        cnt = Counter((CLASSES[a], CLASSES[b]) for a, b in zip(pr["labels"], pred) if a != b)
        print(ds, "most frequent confusions (true -> pred):", cnt.most_common(5))

In [ ]:
# 3.4  are errors linked to image quality? brightness, blur (Laplacian variance), vegetation fraction ----
from scipy.stats import mannwhitneyu

def leaf_mask_hsv(rgb):
    # the paper's leaf mask (Section 2.5): S>25, 20<V<250, close 7x7, open 5x5
    hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)
    s, v = hsv[:, :, 1], hsv[:, :, 2]
    m = ((s > 25) & (v > 20) & (v < 250)).astype(np.uint8) * 255
    m = cv2.morphologyEx(m, cv2.MORPH_CLOSE, np.ones((7, 7), np.uint8))
    m = cv2.morphologyEx(m, cv2.MORPH_OPEN, np.ones((5, 5), np.uint8))
    return (m > 0).astype(np.float32)

def image_quality(rel):
    rgb = np.array(load_rgb(resolve(rel), draft=384))
    if max(rgb.shape[:2]) > 384:
        s = 384 / max(rgb.shape[:2])
        rgb = cv2.resize(rgb, (int(rgb.shape[1] * s), int(rgb.shape[0] * s)))
    hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    green = ((hsv[:, :, 0] > 25) & (hsv[:, :, 0] < 95) & (hsv[:, :, 1] > 40) & (hsv[:, :, 2] > 40)).mean()
    return dict(brightness=float(hsv[:, :, 2].mean()), sharpness=float(cv2.Laplacian(gray, cv2.CV_64F).var()),
                vegetation_fraction=float(green), saturation=float(hsv[:, :, 1].mean()))

if RUN["sanity"]:
    q_rows = []
    for ds in ("plantdoc", "fgvc8"):
        pr = PRED[ds]
        correct = pr["probs"].argmax(1) == pr["labels"]
        q = [image_quality(r) for r in tqdm(pr["paths"], desc=f"quality {ds}")]
        for key in ("brightness", "sharpness", "vegetation_fraction"):
            v = np.array([d[key] for d in q])
            t1, t2 = np.percentile(v, [33.3, 66.7])
            bins = [v <= t1, (v > t1) & (v <= t2), v > t2]
            accs = [correct[b].mean() * 100 if b.sum() else float("nan") for b in bins]
            try:
                pval = mannwhitneyu(v[correct], v[~correct]).pvalue if correct.any() and (~correct).any() else float("nan")
            except ValueError:
                pval = float("nan")
            q_rows.append([ds, key, f"{accs[0]:.1f}", f"{accs[1]:.1f}", f"{accs[2]:.1f}",
                           f"{np.median(v[correct]) if correct.any() else float('nan'):.3g}",
                           f"{np.median(v[~correct]) if (~correct).any() else float('nan'):.3g}", f"{pval:.3g}"])
        save_json([dict(path=p, correct=bool(c), **d) for p, c, d in zip(pr["paths"], correct, q)],
                  os.path.join(SANITY_DIR, f"quality_{ds}.json"))
    put_summary("3.4 zero-shot accuracy vs image properties (ResNet50)",
                "Accuracy (%) in the low / middle / high tercile of each property; Mann-Whitney p compares correct vs wrong images.\n\n" +
                md_table(["dataset", "property", "low", "mid", "high", "median (correct)", "median (wrong)", "p"], q_rows))

In [ ]:
# 3.5  latency of the five paper models ON THIS MACHINE (batch 1) - Reviewer B asked for the exact protocol --
@torch.no_grad()
def measure_latency(model, size, n_warm=3, n_timed=20, with_preprocessing=False, sample_rel=None):
    model.eval().to(DEVICE)
    tf = eval_transforms(size)
    x = torch.randn(1, 3, size, size, device=DEVICE)
    times = []
    for i in range(n_warm + n_timed):
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        if with_preprocessing:
            x = tf(load_rgb(resolve(sample_rel))).unsqueeze(0).to(DEVICE)
        with autocast_ctx():
            model(x)
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        if i >= n_warm:
            times.append((time.perf_counter() - t0) * 1000)
    return float(np.mean(times)), float(np.std(times))

if RUN["sanity"]:
    lat_path = os.path.join(SANITY_DIR, f"latency_{DEVICE.type}.json")
    lat = load_json(lat_path, {})
    if not lat:
        for bb in ALL_BACKBONES:
            m = paper_classifier(bb)
            a = measure_latency(m, INPUT_SIZE[bb])
            b = measure_latency(m, INPUT_SIZE[bb], with_preprocessing=True, sample_rel=PV["test"][0][0])
            lat[bb] = dict(model_only_ms=a, with_preprocessing_ms=b,
                           params_M=sum(p.numel() for p in m.parameters()) / 1e6)
            del m
        lat["_hardware"] = dict(device=str(DEVICE), gpu=torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else None,
                                cpu=platform.processor(), threads=torch.get_num_threads(), torch=torch.__version__,
                                amp=USE_AMP, protocol="batch 1, 3 warm-up + 20 timed passes, synchronize before each timestamp")
        save_json(lat, lat_path)
    rows = [[bb, f"{lat[bb]['params_M']:.2f}", f"{lat[bb]['model_only_ms'][0]:.1f} ± {lat[bb]['model_only_ms'][1]:.1f}",
             f"{lat[bb]['with_preprocessing_ms'][0]:.1f} ± {lat[bb]['with_preprocessing_ms'][1]:.1f}"] for bb in ALL_BACKBONES]
    put_summary(f"3.5 latency on {DEVICE.type}", json.dumps(lat["_hardware"]) + "\n\n" +
                md_table(["backbone", "params (M)", "forward only (ms)", "incl. load+resize (ms)"], rows))

## 4. Frogeye leaf spot (FGVC8) vs black rot (PlantVillage) in feature space

**Reviewer A, Q1 + W2.** The earlier run showed that Frogeye embeddings lie closer to PlantVillage *scab* (0.806) than to *black rot* (0.781). This section repeats that analysis properly:

* two feature extractors: the fine-tuned leakage-free ResNet50 **and** a plain ImageNet ResNet50 (not trained on our classes);
* a **control**: where does FGVC8 *scab* land? If field scab also lands away from PV scab, the problem is the lab-to-field shift in general, not the frogeye mapping;
* nearest-centroid assignment and k-NN label distribution for every FGVC8 class;
* the t-SNE figure (saved this time) and an image panel for a visual comparison.

In [ ]:
# 4.1  features + centroid / k-NN analysis ------------------------------------------------------------
from sklearn.neighbors import NearestNeighbors
from sklearn.manifold import TSNE
TSNE_DIR = os.path.join(RESULTS_ROOT, "tsne")
os.makedirs(TSNE_DIR, exist_ok=True)
TSNE_FIELD_PER_CLASS = 300

@torch.no_grad()
def extract_embeddings(model, samples):
    model.eval().to(DEVICE)
    loader = make_loader(samples, eval_transforms(INPUT_SIZE[model.arch]), EVAL_BATCH)
    out = []
    for x, _, _ in loader:
        with autocast_ctx():
            out.append(embed(model, x.to(DEVICE)).float().cpu().numpy())
    return np.concatenate(out)

def cos_matrix(a, b):
    a = a / (np.linalg.norm(a, axis=1, keepdims=True) + 1e-8)
    b = b / (np.linalg.norm(b, axis=1, keepdims=True) + 1e-8)
    return a @ b.T

if RUN["tsne"]:
    rng = random.Random(0)
    field = []
    for c in range(4):
        v = [s for s in FGVC8_EVAL if s[1] == c]
        rng.shuffle(v)
        field += v[:TSNE_FIELD_PER_CLASS]
    pv = PV["test"]
    pv_y = np.array([s[1] for s in pv]); f_y = np.array([s[1] for s in field])
    extractors = {"finetuned_leakfree": lambda: paper_classifier("resnet50"),
                  "imagenet_only": lambda: build_model("resnet50", pretrained=True).to(DEVICE).eval()}
    feats, tsne_txt = {}, []
    for name, make in extractors.items():
        fp = os.path.join(TSNE_DIR, f"features_{name}.npz")
        if os.path.exists(fp):
            z = np.load(fp); pv_f, f_f = z["pv"], z["field"]
        else:
            m = make()
            pv_f, f_f = extract_embeddings(m, pv), extract_embeddings(m, field)
            np.savez_compressed(fp, pv=pv_f, field=f_f)
            del m
        feats[name] = (pv_f, f_f)
        cents = np.stack([pv_f[pv_y == c].mean(0) for c in range(4)])
        sim = np.stack([cos_matrix(f_f[f_y == c], cents).mean(0) for c in range(4)])   # rows: FGVC8 class
        nc = cos_matrix(f_f, cents).argmax(1)                                           # nearest PV centroid
        knn = NearestNeighbors(n_neighbors=min(10, len(pv_f))).fit(pv_f)
        _, idx = knn.kneighbors(f_f)
        rows = []
        for c in range(4):
            sel = f_y == c
            knn_dist = np.bincount(pv_y[idx[sel]].ravel(), minlength=4) / max(idx[sel].size, 1)
            nc_dist = np.bincount(nc[sel], minlength=4) / max(sel.sum(), 1)
            rows.append([f"FGVC8 {CLASS_TITLES[c]}" + (" (= frogeye)" if c == 1 else "")] +
                        [f"{sim[c, j]:.3f}" for j in range(4)] +
                        [f"{nc_dist[c]*100:.0f}%", CLASS_TITLES[int(nc_dist.argmax())], f"{knn_dist[c]*100:.0f}%"])
        tsne_txt.append(f"**{name}** (mean cosine similarity to each PlantVillage class centroid)\n\n" +
                        md_table(["field class"] + [f"PV {t}" for t in CLASS_TITLES] +
                                 ["nearest-centroid = own class", "most common nearest centroid", "10-NN = own class"], rows))
        save_json(dict(similarity=sim, nearest_centroid=nc, field_labels=f_y), os.path.join(TSNE_DIR, f"stats_{name}.json"))
    put_summary("4.1 frogeye mapping in feature space", "\n\n".join(tsne_txt))

In [ ]:
# 4.2  t-SNE figure + image panel --------------------------------------------------------------------
if RUN["tsne"]:
    pv_f, f_f = feats["finetuned_leakfree"]
    keep_field = np.isin(f_y, [0, 1])      # FGVC8 scab (control) + FGVC8 frogeye
    X = np.concatenate([pv_f, f_f[keep_field]])
    perp = max(2, min(30, (len(X) - 1) // 3))
    emb = TSNE(n_components=2, perplexity=perp, init="pca", learning_rate="auto", random_state=0).fit_transform(X)
    e_pv, e_f = emb[:len(pv_f)], emb[len(pv_f):]
    fy = f_y[keep_field]
    colors = ["#E69F00", "#D55E00", "#009E73", "#0072B2"]
    fig, ax = plt.subplots(figsize=(7.5, 6.5))
    for c in range(4):
        ax.scatter(e_pv[pv_y == c, 0], e_pv[pv_y == c, 1], s=12, c=colors[c], alpha=0.7, label=f"PlantVillage {CLASS_TITLES[c]}")
    ax.scatter(e_f[fy == 1, 0], e_f[fy == 1, 1], s=22, marker="x", c="black", linewidths=0.8, label="FGVC8 frogeye leaf spot")
    ax.scatter(e_f[fy == 0, 0], e_f[fy == 0, 1], s=22, marker="+", c="grey", linewidths=0.8, label="FGVC8 scab (control)")
    ax.set_xlabel("t-SNE 1"); ax.set_ylabel("t-SNE 2"); ax.legend(fontsize=8, loc="best")
    ax.set_title("Leakage-free ResNet50 embeddings (PlantVillage test vs FGVC8 field)")
    save_fig(fig, "fig_tsne_frogeye_vs_blackrot"); plt.show()

    # visual panel: PV black rot, FGVC8 frogeye, PV scab, FGVC8 scab
    groups = [("PlantVillage black rot", [s for s in pv if s[1] == 1]),
              ("FGVC8 frogeye leaf spot", [s for s in field if s[1] == 1]),
              ("PlantVillage scab", [s for s in pv if s[1] == 0]),
              ("FGVC8 scab", [s for s in field if s[1] == 0])]
    fig, axes = plt.subplots(4, 5, figsize=(12, 10))
    for r, (title, items) in enumerate(groups):
        pick = random.Random(r).sample(items, min(5, len(items)))
        for c in range(5):
            ax = axes[r, c]; ax.axis("off")
            if c < len(pick):
                ax.imshow(load_rgb(resolve(pick[c][0]), draft=384))
        axes[r, 0].set_title(title, loc="left", fontsize=10)
    save_fig(fig, "fig_frogeye_blackrot_panel"); plt.show()

## 5. Grad-CAM per class + an independent leaf mask

**Reviewer A, Q5; Reviewer B (validate CAM-in-leaf, per-class scores, more examples).**

For all 490 test images: Grad-CAM (ResNet50, predicted class, own implementation that follows `pytorch-grad-cam`), the paper's HSV leaf mask, and an **independent GrabCut mask** (OpenCV, colour/graph-cut based, no thresholds shared with the HSV mask). If an image scores low with the HSV mask but high with GrabCut, the low score comes from the **mask**, not from the model looking at the background.

In [ ]:
# 5.1  Grad-CAM, HSV mask, GrabCut mask ------------------------------------------------------------------
class GradCAM:
    def __init__(self, model, layer):
        self.model, self.acts, self.grads = model, None, None
        def fwd_hook(module, inp, out):
            self.acts = out
            if out.requires_grad:
                out.register_hook(lambda g: setattr(self, "grads", g))
        layer.register_forward_hook(fwd_hook)
    def __call__(self, x, size):
        self.model.zero_grad(set_to_none=True)
        out = self.model(x)
        idx = out.argmax(1)
        out.gather(1, idx[:, None]).sum().backward()
        w = self.grads.mean(dim=(2, 3), keepdim=True)
        cam = F.relu((w * self.acts).sum(1)).detach().cpu().numpy()
        res = []
        for c in cam:
            c = c - c.min()
            c = c / (c.max() + 1e-7)
            res.append(cv2.resize(c, (size, size), interpolation=cv2.INTER_LINEAR))
        return np.stack(res), idx.cpu().numpy(), torch.softmax(out.detach().float(), 1).cpu().numpy()

def grabcut_mask(rgb, margin=2, iters=5):
    bgr = cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)
    mask = np.zeros(bgr.shape[:2], np.uint8)
    h, w = mask.shape
    bgd, fgd = np.zeros((1, 65), np.float64), np.zeros((1, 65), np.float64)
    cv2.grabCut(bgr, mask, (margin, margin, w - 2 * margin, h - 2 * margin), bgd, fgd, iters, cv2.GC_INIT_WITH_RECT)
    return ((mask == cv2.GC_FGD) | (mask == cv2.GC_PR_FGD)).astype(np.float32)

def cam_in_mask(cam, mask):
    return float((cam * mask).sum() / (cam.sum() + 1e-8))

def iou(a, b):
    a, b = a > 0.5, b > 0.5
    u = np.logical_or(a, b).sum()
    return float(np.logical_and(a, b).sum() / u) if u else float("nan")

CAM_DIR = os.path.join(RESULTS_ROOT, "gradcam")
os.makedirs(CAM_DIR, exist_ok=True)
CAM_JSON = os.path.join(CAM_DIR, "per_image.json")

def cam_records(samples, model=None):
    model = model or paper_classifier("resnet50")
    for p in model.parameters():
        p.requires_grad_(True)
    gc = GradCAM(model, body_of(model)[-1])      # last ResNet50 block (layer4)
    size, tf = 224, eval_transforms(224)
    recs = []
    for i in tqdm(range(0, len(samples), CAM_BATCH), desc="Grad-CAM"):
        chunk = samples[i:i + CAM_BATCH]
        imgs = [load_rgb(resolve(r)) for r, _ in chunk]
        x = torch.stack([tf(im) for im in imgs]).to(DEVICE)
        cams, preds, probs = gc(x, size)
        for (r, y), im, cam, pd_, pb in zip(chunk, imgs, cams, preds, probs):
            rgb = np.array(im.resize((size, size), Image.BILINEAR))
            mh, mg = leaf_mask_hsv(rgb), grabcut_mask(rgb)
            recs.append(dict(path=r, label=int(y), pred=int(pd_), conf=float(pb.max()),
                             score_hsv=cam_in_mask(cam, mh), score_grabcut=cam_in_mask(cam, mg),
                             area_hsv=float(mh.mean()), area_grabcut=float(mg.mean()), iou_hsv_grabcut=iou(mh, mg)))
    return recs

if RUN["gradcam"]:
    if not os.path.exists(CAM_JSON):
        save_json(cam_records(PV["test"]), CAM_JSON)
    CAM = load_json(CAM_JSON)
    LOW = 0.60
    rows = []
    for c in ALL4 + [None]:
        R = [r for r in CAM if c is None or r["label"] == c]
        low = [r for r in R if r["score_hsv"] < LOW]
        mask_fail = sum(r["score_grabcut"] >= 0.80 for r in low)
        off_leaf = sum(r["score_grabcut"] < LOW for r in low)
        rows.append(["all (pooled)" if c is None else CLASS_TITLES[c], len(R),
                     fmt_ms([r["score_hsv"] for r in R], 100, 1), fmt_ms([r["score_grabcut"] for r in R], 100, 1),
                     fmt_ms([r["iou_hsv_grabcut"] for r in R], 1, 2), len(low), mask_fail, off_leaf])
    put_summary("5.1 CAM-in-leaf per class",
                f"CAM-in-leaf (%) with the paper's HSV mask and with an independent GrabCut mask (n = test images).\n"
                f"'low' = HSV score < {LOW:.2f}; of those, 'mask failure' = GrabCut score >= 0.80, 'off-leaf' = GrabCut score < {LOW:.2f}.\n\n" +
                md_table(["class", "n", "CAM-in-leaf HSV", "CAM-in-leaf GrabCut", "IoU(HSV, GrabCut)", "low", "mask failure", "off-leaf"], rows))

In [ ]:
# 5.2  figures: lowest-scoring images per class (with both mask outlines) and typical examples incl. healthy
def overlay(rgb, cam):
    heat = cv2.applyColorMap(np.uint8(255 * cam), cv2.COLORMAP_JET)[:, :, ::-1]
    return np.uint8(0.5 * rgb + 0.5 * heat)

def contour(img, mask, color):
    cs, _ = cv2.findContours((mask > 0.5).astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    out = img.copy()
    cv2.drawContours(out, cs, -1, color, 2)
    return out

def cam_panel(records, title, fname, ncols=4):
    if not records:
        return
    model = paper_classifier("resnet50")
    gc = GradCAM(model, body_of(model)[-1])
    tf = eval_transforms(224)
    fig, axes = plt.subplots(2, ncols, figsize=(3 * ncols, 6.4))
    axes = np.array(axes).reshape(2, ncols)
    for ax in axes.ravel():
        ax.axis("off")
    for j, r in enumerate(records[:ncols]):
        im = load_rgb(resolve(r["path"]))
        rgb = np.array(im.resize((224, 224), Image.BILINEAR))
        cam, _, _ = gc(tf(im).unsqueeze(0).to(DEVICE), 224)
        img = contour(contour(rgb, leaf_mask_hsv(rgb), (255, 0, 0)), grabcut_mask(rgb), (0, 255, 255))
        axes[0, j].imshow(img)
        axes[0, j].set_title(f"{CLASS_TITLES[r['label']]}\nHSV {r['score_hsv']:.2f} | GrabCut {r['score_grabcut']:.2f}", fontsize=8)
        axes[1, j].imshow(overlay(rgb, cam[0]))
    fig.suptitle(title + "   (red = HSV leaf mask, cyan = GrabCut mask)", fontsize=10)
    save_fig(fig, fname); plt.show()

if RUN["gradcam"]:
    for c in ALL4:
        worst = sorted([r for r in CAM if r["label"] == c], key=lambda r: r["score_hsv"])
        cam_panel(worst[:4], f"Lowest CAM-in-leaf: {CLASS_TITLES[c]}", f"fig_gradcam_lowest_{CLASSES[c]}")
    typical = []
    for c in ALL4:
        R = sorted([r for r in CAM if r["label"] == c and r["pred"] == c], key=lambda r: r["score_hsv"])
        if R:
            typical.append(R[len(R) // 2])      # median-scoring correct image of each class
    cam_panel(typical, "Typical (median-score) Grad-CAM per class", "fig_gradcam_typical_per_class")

## 6. Manual masks (optional, Reviewer B): prepare and score

1. Run the next cell once. It copies **24 test images** (6 per class: the 3 lowest CAM-in-leaf + 3 random) into `manual_annotations/images`.
2. Draw the masks with LabelMe (about 30–45 minutes in total):
   ```
   pip install labelme
   labelme manual_annotations/images --labels leaf,lesion --nodata --autosave
   ```
   For each image, draw one polygon with the label **leaf** around the whole leaf, and polygons with the label **lesion** around the visible lesions (skip lesions on healthy leaves). LabelMe saves a `.json` file next to each image.
3. Run the scoring cell. With the manual leaf masks, CAM-in-leaf is recomputed and the HSV and GrabCut masks are compared with a human mask. With the lesion masks, the Grad-CAM and colour-detector IoU/Dice and a *pointing game* (does the hottest pixel fall on a lesion?) are computed.

If no masks are drawn, the scoring cell just says so. The GrabCut comparison in Section 5 already answers the reviewer without manual work.

In [ ]:
# 6.1  prepare the images to annotate --------------------------------------------------------------
MAN_IMG = os.path.join(MANUAL_DIR, "images")
MAN_MASK = os.path.join(MANUAL_DIR, "masks")
os.makedirs(MAN_IMG, exist_ok=True); os.makedirs(MAN_MASK, exist_ok=True)
if RUN["manual_masks"] and os.path.exists(CAM_JSON):
    CAM = load_json(CAM_JSON)
    existing = [f for f in os.listdir(MAN_IMG) if f.lower().endswith((".jpg", ".png", ".jpeg"))]
    if not existing:
        rng = random.Random(7)
        for c in ALL4:
            R = sorted([r for r in CAM if r["label"] == c], key=lambda r: r["score_hsv"])
            pick = R[:3] + rng.sample(R[3:], min(3, len(R) - 3))
            for r in pick:
                shutil.copyfile(resolve(r["path"]), os.path.join(MAN_IMG, f"{CLASSES[c]}__{os.path.basename(r['path'])}"))
        print("copied", len(os.listdir(MAN_IMG)), "images to", MAN_IMG)
    else:
        print(len(existing), "images already in", MAN_IMG)

In [ ]:
# 6.2  score the manual masks (LabelMe .json next to the image, or PNG masks <stem>_leaf.png / <stem>_lesion.png)
def masks_from_labelme(json_path, size=224):
    d = load_json(json_path)
    W, H = d["imageWidth"], d["imageHeight"]
    out = {}
    for lab in ("leaf", "lesion"):
        m = Image.new("L", (W, H), 0)
        dr = ImageDraw.Draw(m)
        n = 0
        for sh in d.get("shapes", []):
            if sh.get("label", "").strip().lower() != lab:
                continue
            pts = [tuple(p) for p in sh["points"]]
            if sh.get("shape_type") == "rectangle" and len(pts) == 2:
                dr.rectangle(pts, fill=255)
            elif sh.get("shape_type") == "circle" and len(pts) == 2:
                (cx, cy), (px, py) = pts
                rad = math.hypot(px - cx, py - cy)
                dr.ellipse([cx - rad, cy - rad, cx + rad, cy + rad], fill=255)
            elif len(pts) >= 3:
                dr.polygon(pts, fill=255)
            n += 1
        out[lab] = (np.array(m.resize((size, size), Image.NEAREST)) > 127).astype(np.float32) if n else None
    return out

def color_lesion_mask(rgb):
    # the colour-based lesion detector used for auto_vs_reference.csv (ExG + Otsu, dark pixels inside the leaf)
    bgr = cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)
    leaf = leaf_mask_hsv(rgb) > 0
    b, g, r = cv2.split(bgr.astype(np.float32))
    exg = cv2.normalize(2 * g - r - b, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    _, les = cv2.threshold(exg, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    L = cv2.cvtColor(bgr, cv2.COLOR_BGR2LAB)[:, :, 0]
    dark = (L < np.percentile(L[leaf], 40)) if leaf.any() else np.zeros_like(leaf)
    return ((les > 0) & dark & leaf).astype(np.float32)

def dice(a, b):
    a, b = a > 0.5, b > 0.5
    s = a.sum() + b.sum()
    return float(2 * np.logical_and(a, b).sum() / s) if s else float("nan")

if RUN["manual_masks"]:
    items = []
    for f in sorted(os.listdir(MAN_IMG)):
        stem, ext = os.path.splitext(f)
        if ext.lower() not in (".jpg", ".jpeg", ".png"):
            continue
        masks = {"leaf": None, "lesion": None}
        js = os.path.join(MAN_IMG, stem + ".json")
        if os.path.exists(js):
            masks = masks_from_labelme(js)
        for lab in ("leaf", "lesion"):
            p = os.path.join(MAN_MASK, f"{stem}_{lab}.png")
            if masks[lab] is None and os.path.exists(p):
                masks[lab] = (np.array(Image.open(p).convert("L").resize((224, 224), Image.NEAREST)) > 127).astype(np.float32)
        if masks["leaf"] is not None or masks["lesion"] is not None:
            items.append((os.path.join(MAN_IMG, f), stem.split("__")[0], masks))
    if not items:
        print("No manual masks found yet - draw them with LabelMe (see the instructions above) and run this cell again.")
    else:
        model = paper_classifier("resnet50")
        gc = GradCAM(model, body_of(model)[-1])
        tf = eval_transforms(224)
        rec = []
        for path, cls, masks in items:
            im = load_rgb(path)
            rgb = np.array(im.resize((224, 224), Image.BILINEAR))
            cam = gc(tf(im).unsqueeze(0).to(DEVICE), 224)[0][0]
            r = dict(image=os.path.basename(path), cls=cls)
            mh, mg = leaf_mask_hsv(rgb), grabcut_mask(rgb)
            if masks["leaf"] is not None:
                ml = masks["leaf"]
                r.update(cam_in_leaf_manual=cam_in_mask(cam, ml), cam_in_leaf_hsv=cam_in_mask(cam, mh),
                         cam_in_leaf_grabcut=cam_in_mask(cam, mg), iou_hsv_manual=iou(mh, ml), iou_grabcut_manual=iou(mg, ml))
            if masks["lesion"] is not None and masks["lesion"].sum() > 0:
                ms = masks["lesion"]
                cam_u8 = np.uint8(cam * 255)
                _, cb = cv2.threshold(cam_u8, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
                cb = (cb > 0).astype(np.float32)
                cl = color_lesion_mask(rgb)
                y0, x0 = np.unravel_index(cam.argmax(), cam.shape)
                dil = cv2.dilate(ms, np.ones((11, 11), np.uint8))
                r.update(iou_cam_lesion=iou(cb, ms), dice_cam_lesion=dice(cb, ms), iou_color_lesion=iou(cl, ms),
                         dice_color_lesion=dice(cl, ms), cam_energy_in_lesion=cam_in_mask(cam, ms),
                         lesion_area_fraction=float(ms.mean()), pointing_game_hit=float(dil[y0, x0] > 0))
            rec.append(r)
        save_json(rec, os.path.join(CAM_DIR, "manual_mask_scores.json"))
        keys = ["cam_in_leaf_manual", "cam_in_leaf_hsv", "cam_in_leaf_grabcut", "iou_hsv_manual", "iou_grabcut_manual",
                "iou_cam_lesion", "dice_cam_lesion", "iou_color_lesion", "dice_color_lesion", "cam_energy_in_lesion",
                "lesion_area_fraction", "pointing_game_hit"]
        rows = []
        for k in keys:
            v = [r[k] for r in rec if k in r and not (isinstance(r[k], float) and math.isnan(r[k]))]
            if v:
                rows.append([k, len(v), fmt_ms(v, 1, 3)])
        put_summary("6.2 manual mask validation", md_table(["metric", "n", "mean ± SD"], rows))

## 7. YOLOv8s vs YOLO11s (Reviewer A, Q3)

The comparison already ran (3 seeds per model, 30 epochs, same PlantDoc split). The first cell summarises it. The optional second cell re-trains at 100 epochs, like the paper's detector; it needs `ultralytics` and resumes interrupted runs.

In [ ]:
# 7.1  summarise the finished YOLO comparison ---------------------------------------------------------
YOLO_DIR = os.path.join(RESULTS_ROOT, "yolo")
os.makedirs(YOLO_DIR, exist_ok=True)

def yolo_summary(raw, label):
    rows = []
    for key in ("yolov8s", "yolo11s"):
        runs = raw.get(key, [])
        if not runs:
            continue
        for kind in ("base", "tta"):
            rows.append([key, kind, len(runs)] + [fmt_ms([r[kind][m] for r in runs], 1, 3)
                                                   for m in ("precision", "recall", "map50", "map50_95")])
    return f"{label}\n\n" + md_table(["model", "eval", "seeds", "precision", "recall", "mAP@0.5", "mAP@0.5:0.95"], rows)

if RUN["yolo_import"]:
    p = os.path.join(IMPORT_DIR, "yolo_comparison_raw_30ep.json")
    if os.path.exists(p):
        put_summary("7.1 YOLOv8s vs YOLO11s (30 epochs, imported)",
                    yolo_summary(load_json(p), "PlantDoc apple test split (29 images, 34 boxes); mean ± SD over seeds."))
    else:
        print("old YOLO results not found in", OLD_RESULT_ZIPS)

In [ ]:
# 7.2  (optional) re-train both detectors at 100 epochs -----------------------------------------------
YOLO_EPOCHS = 100 if not FAST_DEV_RUN else 1
if RUN["yolo_train"]:
    from ultralytics import YOLO
    yaml_path = os.path.join(YOLO_DIR, "plantdoc_apple_local.yaml")
    with open(yaml_path, "w", encoding="utf-8") as f:
        f.write(f"path: {YOLO_ROOT}\ntrain: images/train\nval: images/val\ntest: images/val\nnc: 3\n"
                f"names: ['Apple_Scab_Leaf', 'Apple_rust_leaf', 'Apple_leaf']\n")
    raw_p = os.path.join(YOLO_DIR, f"yolo_comparison_raw_{YOLO_EPOCHS}ep.json")
    raw = load_json(raw_p, {"yolov8s": [], "yolo11s": []})
    for weights, key in [("yolov8s.pt", "yolov8s"), ("yolo11s.pt", "yolo11s")]:
        for seed in SEEDS:
            if any(r["seed"] == seed for r in raw[key]):
                continue
            name = f"{key}_seed{seed}_{YOLO_EPOCHS}ep"
            last = os.path.join(YOLO_DIR, "runs", name, "weights", "last.pt")
            if os.path.exists(last):
                model = YOLO(last); model.train(resume=True)          # continue an interrupted run
            else:
                model = YOLO(weights)
                model.train(data=yaml_path, epochs=YOLO_EPOCHS, imgsz=640, batch=16 if DEVICE.type == "cuda" else 8,
                            cos_lr=True, seed=seed, project=os.path.join(YOLO_DIR, "runs"), name=name, exist_ok=True,
                            mosaic=1.0, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, verbose=False,
                            device=0 if DEVICE.type == "cuda" else "cpu", workers=0)
            best = YOLO(os.path.join(YOLO_DIR, "runs", name, "weights", "best.pt"))
            pack = lambda m: dict(precision=float(m.box.mp), recall=float(m.box.mr), map50=float(m.box.map50), map50_95=float(m.box.map))
            res = dict(seed=seed, base=pack(best.val(data=yaml_path, split="test", imgsz=640, workers=0)),
                       tta=pack(best.val(data=yaml_path, split="test", imgsz=640, augment=True, workers=0)))
            raw[key].append(res); save_json(raw, raw_p)
            log(f"YOLO {key} seed {seed}: mAP50 base={res['base']['map50']:.3f} tta={res['tta']['map50']:.3f}")
    put_summary(f"7.2 YOLOv8s vs YOLO11s ({YOLO_EPOCHS} epochs)", yolo_summary(raw, f"{YOLO_EPOCHS} epochs"))

## 8. TRAINING: data efficiency for more backbones (Reviewer A W5, Reviewer B Methodology)

Same protocol as Table 9: the fixed FGVC8 pool (1,600 train + 400 test, 500/class), fractions 5–100%, 3 seeds, head always re-initialised, and two initialisations: **ImageNet** and the **leakage-free lab backbone** (the paper checkpoint of the *same* architecture).

* ResNet50 is run again with this code as the reference, so every backbone is compared under identical conditions.
* `DE_INCLUDE_LEAKY_INIT = True` adds a third ResNet50 start: the backbone trained on the **leaky** split (Section 9). This directly tests Reviewer A's point that *the honest model adapts faster than the leaky one*. Those runs are skipped until Section 9 has saved its weights, so run this section again after Section 9.

Every finished run is stored immediately in `result_new/data_efficiency/results.json`. Stop at any time; the next run starts where it stopped.

In [ ]:
# 8.1  run the sweep ----------------------------------------------------------------------------------
DE_DIR = os.path.join(RESULTS_ROOT, "data_efficiency")
LEAKY_DIR = os.path.join(RESULTS_ROOT, "leaky_split")
LEAKY_WEIGHTS = os.path.join(LEAKY_DIR, "weights", "leaky_resnet50_seed0.pt")
de_store = ResultStore(os.path.join(DE_DIR, "results.json"))
pool_labels = np.array([s[1] for s in FIELD_TRAIN_POOL])

def stratified_subset(labels, frac, seed):
    if frac >= 1.0:
        return np.arange(len(labels))
    keep, _ = train_test_split(np.arange(len(labels)), train_size=frac, random_state=seed, stratify=labels)
    return keep

def inner_val_split(idx, labels, seed, frac=0.15):
    n_val = max(len(CLASSES), int(round(frac * len(idx))))
    return train_test_split(idx, test_size=n_val, random_state=seed, stratify=labels[idx])

def de_inits(bb):
    inits = ["imagenet", "lab"]
    if bb == "resnet50" and DE_INCLUDE_LEAKY_INIT:
        inits.append("leaky")
    return inits

if RUN["data_eff"]:
    for bb in DE_BACKBONES:
        for init in de_inits(bb):
            if init == "leaky" and not os.path.exists(LEAKY_WEIGHTS):
                print("skip init=leaky (run Section 9 first, then run this cell again)")
                continue
            for frac in DE_FRACTIONS:
                for seed in SEEDS:
                    if de_store.has(backbone=bb, init=init, fraction=frac, seed=seed):
                        continue
                    set_seed(seed)
                    sub = stratified_subset(pool_labels, frac, seed)
                    tr_idx, va_idx = inner_val_split(sub, pool_labels, seed)
                    model = build_model(bb, pretrained=(init == "imagenet"))
                    if init == "lab":
                        transfer_body(model, os.path.join(CKPT_ROOT, CKPT_FILES[bb]))
                    elif init == "leaky":
                        transfer_body(model, LEAKY_WEIGHTS)
                    tag = f"[DE {bb} {init} {int(frac*100)}% s{seed}]"
                    run_dir = os.path.join(DE_DIR, "runs", f"{bb}_{init}_f{frac}_s{seed}")

                    def _de_job():
                        t0 = time.time()
                        mdl, info = fit_two_phase(model, [FIELD_TRAIN_POOL[i] for i in tr_idx],
                                                  [FIELD_TRAIN_POOL[i] for i in va_idx], run_dir, seed, tag=tag)
                        m, _ = evaluate(mdl, FIELD_TEST)
                        de_store.add(dict(backbone=bb, init=init, fraction=frac, seed=seed, n_train=len(tr_idx),
                                          n_val=len(va_idx), macro_f1=m["f1_macro"], accuracy=m["accuracy"],
                                          per_class_f1={k: v["f1"] for k, v in m["per_class"].items()},
                                          best_epoch=info["best_epoch"], micro_batch=MICRO_BATCH,
                                          minutes=round((time.time() - t0) / 60, 1)))
                        log(f"{tag} n_train={len(tr_idx)} macro-F1={m['f1_macro']:.4f} ({(time.time()-t0)/60:.1f} min)")
                        cleanup_run(run_dir)

                    oom_guard(tag, _de_job)
                    del model
                    if DEVICE.type == "cuda":
                        torch.cuda.empty_cache()

In [ ]:
# 8.2  tables (Table 9 layout), Wilcoxon test, field images needed for a target F1, figure ----------------
from scipy.stats import wilcoxon

def de_table(bb):
    inits = [i for i in de_inits(bb) if de_store.get(backbone=bb, init=i)]
    rows, paired = [], defaultdict(list)
    for frac in DE_FRACTIONS:
        cell = {i: {r["seed"]: r["macro_f1"] for r in de_store.get(backbone=bb, init=i, fraction=frac)} for i in inits}
        n_img = [r["n_train"] for r in de_store.get(backbone=bb, fraction=frac)]
        row = [f"{int(frac*100)}% ({n_img[0] if n_img else '?'} img)"]
        for i in inits:
            row.append(fmt_ms(list(cell[i].values())) if cell[i] else "-")
        if "imagenet" in cell and "lab" in cell and cell["imagenet"] and cell["lab"]:
            row.append(f"{np.mean(list(cell['lab'].values())) - np.mean(list(cell['imagenet'].values())):+.3f}")
            for s in set(cell["imagenet"]) & set(cell["lab"]):
                paired["lab_vs_imagenet"].append((cell["lab"][s], cell["imagenet"][s]))
        else:
            row.append("-")
        if "leaky" in cell and cell.get("leaky") and cell.get("lab"):
            row.append(f"{np.mean(list(cell['lab'].values())) - np.mean(list(cell['leaky'].values())):+.3f}")
            for s in set(cell["leaky"]) & set(cell["lab"]):
                paired["lab_vs_leaky"].append((cell["lab"][s], cell["leaky"][s]))
        elif "leaky" in inits:
            row.append("-")
        rows.append(row)
    header = ["field data"] + [{"imagenet": "ImageNet-init", "lab": "Lab-init (leak-free)", "leaky": "Leaky-lab-init"}[i]
                               for i in inits] + ["gain lab-ImageNet"] + (["gain lab-leaky"] if "leaky" in inits else [])
    tests = []
    for k, pairs in paired.items():
        a, b = np.array(pairs).T
        wins = int((a > b).sum())
        try:
            p = wilcoxon(a, b, alternative="greater").pvalue
        except ValueError:
            p = float("nan")
        tests.append(f"{k}: lab better in {wins}/{len(a)} paired (fraction, seed) runs, one-sided Wilcoxon p = {p:.3g}")
    return md_table(header, rows), tests

def images_to_reach(bb, init, target):
    pts = []
    for frac in DE_FRACTIONS:
        R = de_store.get(backbone=bb, init=init, fraction=frac)
        if R:
            pts.append((np.mean([r["n_train"] for r in R]), np.mean([r["macro_f1"] for r in R])))
    for (n0, f0), (n1, f1) in zip(pts, pts[1:]):
        if f0 >= target:
            return n0
        if f1 >= target:
            return n0 + (target - f0) * (n1 - n0) / (f1 - f0)
    return pts[0][0] if pts and pts[0][1] >= target else None

if RUN["data_eff"] and de_store.rows:
    txt = []
    for bb in DE_BACKBONES:
        if not de_store.get(backbone=bb):
            continue
        t, tests = de_table(bb)
        need = []
        for target in (0.85, 0.90, 0.95):
            vals = {i: images_to_reach(bb, i, target) for i in de_inits(bb)}
            need.append(f"macro-F1 {target:.2f}: " + ", ".join(f"{i} ~{v:.0f} img" if v else f"{i} not reached"
                                                               for i, v in vals.items()))
        txt.append(f"**{bb}** (FGVC8 field test n={len(FIELD_TEST)}, macro-F1 mean ± SD over seeds)\n\n{t}\n" +
                   "\n".join("- " + s for s in tests + need))
    put_summary("8.2 data efficiency", "\n\n".join(txt))

    bbs = [bb for bb in DE_BACKBONES if de_store.get(backbone=bb)]
    fig, axes = plt.subplots(1, len(bbs), figsize=(4.6 * len(bbs), 3.8), squeeze=False)
    style = {"imagenet": ("ImageNet-init", "#0072B2", "o"), "lab": ("Leakage-free lab-init", "#D55E00", "s"),
             "leaky": ("Leaky lab-init", "#7F7F7F", "^")}
    for ax, bb in zip(axes[0], bbs):
        for init in de_inits(bb):
            xs, ms, ss = [], [], []
            for frac in DE_FRACTIONS:
                v = [r["macro_f1"] for r in de_store.get(backbone=bb, init=init, fraction=frac)]
                if v:
                    xs.append(frac * 100); ms.append(np.mean(v)); ss.append(np.std(v, ddof=1) if len(v) > 1 else 0)
            if xs:
                lab, col, mk = style[init]
                ax.errorbar(xs, ms, yerr=ss, label=lab, color=col, marker=mk, capsize=3, lw=1.6)
        ax.set_xscale("log"); ax.set_xticks([5, 10, 25, 50, 100]); ax.set_xticklabels(["5", "10", "25", "50", "100"])
        ax.set_xlabel("field training data (%)"); ax.set_ylabel("macro-F1 (FGVC8 field test)")
        ax.set_title(bb); ax.grid(alpha=0.3)
    axes[0][0].legend(fontsize=8)
    save_fig(fig, "fig_data_efficiency_backbones"); plt.show()

## 9. TRAINING: leaky split vs honest split (Reviewer A, W3)

* **Leaky model:** ResNet50 trained on a random image-level split of the raw 9,819-file folder (rotated/flipped copies of one leaf land on both sides), exactly like the old script, 3 seeds.
* The leaky test set is split into **contaminated** images (their leaf is also in training) and **clean** images (leaf never seen). The gap between the two is the inflation caused by leakage.
* **Honest model:** the same code trained on the leakage-free split (3 seeds), so both rows are comparable.
* Both are tested zero-shot on PlantDoc and FGVC8, and the leaky backbone is also used in the data-efficiency study (Section 8, `init = leaky`).

In [ ]:
# 9.1  leaky and honest training ------------------------------------------------------------------------
leaky_store = ResultStore(os.path.join(LEAKY_DIR, "results.json"))
LEAKY_LABELS = np.array([s[1] for s in LEAKY_ALL])

def naive_random_split(labels, seed, ratios=(0.70, 0.15, 0.15)):
    # identical to the old 02 script: IMAGE-level stratified split (this is what creates the leak)
    idx = np.arange(len(labels))
    tr, rest = train_test_split(idx, test_size=(1 - ratios[0]), random_state=seed, stratify=labels)
    va, te = train_test_split(rest, test_size=(1 - ratios[1] / (ratios[1] + ratios[2])), random_state=seed,
                              stratify=labels[rest])
    return tr, va, te

def field_eval(model):
    out = {}
    for ds, samples, present in [("plantdoc", PLANTDOC, PLANTDOC_PRESENT), ("fgvc8", FGVC8_EVAL, ALL4)]:
        m, _ = evaluate(model, samples, present)
        out[ds] = dict(accuracy=m["accuracy"], restricted_accuracy=m["restricted_accuracy"], f1_macro=m["f1_macro"])
    return out

def _leaky_job(seed):
        set_seed(seed)
        tr, va, te = naive_random_split(LEAKY_LABELS, seed)
        train_leaves = set(LEAKY_BASE[i] for i in tr)
        contaminated = np.array([LEAKY_BASE[i] in train_leaves for i in te])
        val_contam = float(np.mean([LEAKY_BASE[i] in train_leaves for i in va]))
        test_leaves = set(LEAKY_BASE[i] for i in te)
        run_dir = os.path.join(LEAKY_DIR, "runs", f"leaky_s{seed}")
        model = build_model("resnet50", pretrained=True)
        model, info = fit_two_phase(model, [LEAKY_ALL[i] for i in tr], [LEAKY_ALL[i] for i in va], run_dir, seed,
                                    tag=f"[leaky s{seed}]")
        _, pr = evaluate(model, [LEAKY_ALL[i] for i in te])
        correct = pr["probs"].argmax(1) == pr["labels"]
        row = dict(model="leaky", seed=seed, n_train=len(tr), n_test=len(te),
                   test_images_contaminated_pct=float(contaminated.mean() * 100),
                   val_images_contaminated_pct=val_contam * 100,
                   test_leaves_seen_in_train_pct=float(np.mean([l in train_leaves for l in test_leaves]) * 100),
                   acc_test_all=float(correct.mean()), acc_test_contaminated=float(correct[contaminated].mean()) if contaminated.any() else None,
                   acc_test_clean=float(correct[~contaminated].mean()) if (~contaminated).any() else None,
                   n_test_clean=int((~contaminated).sum()),
                   f1_test_all=metrics_from(pr["labels"], pr["probs"], ALL4)["f1_macro"],
                   best_epoch=info["best_epoch"], **{f"zs_{k}": v for k, v in field_eval(model).items()})
        leaky_store.add(row)
        log(f"[leaky s{seed}] test acc={row['acc_test_all']:.4f} (contaminated {row['test_images_contaminated_pct']:.1f}% "
            f"-> {row['acc_test_contaminated']}, clean -> {row['acc_test_clean']}) PD={row['zs_plantdoc']['accuracy']:.3f} "
            f"FGVC8={row['zs_fgvc8']['accuracy']:.3f}")
        cleanup_run(run_dir, keep_weights_as=os.path.join(LEAKY_DIR, "weights", f"leaky_resnet50_seed{seed}.pt"))

def _honest_job(seed):
        set_seed(seed)
        run_dir = os.path.join(LEAKY_DIR, "runs", f"honest_s{seed}")
        model = build_model("resnet50", pretrained=True)
        model, info = fit_two_phase(model, PV["train"], PV["val"], run_dir, seed, tag=f"[honest s{seed}]")
        m, _ = evaluate(model, PV["test"])
        row = dict(model="honest", seed=seed, n_train=len(PV["train"]), n_test=len(PV["test"]),
                   test_images_contaminated_pct=0.0, acc_test_all=m["accuracy"], acc_test_clean=m["accuracy"],
                   f1_test_all=m["f1_macro"], best_epoch=info["best_epoch"],
                   **{f"zs_{k}": v for k, v in field_eval(model).items()})
        leaky_store.add(row)
        log(f"[honest s{seed}] test acc={m['accuracy']:.4f} PD={row['zs_plantdoc']['accuracy']:.3f} FGVC8={row['zs_fgvc8']['accuracy']:.3f}")
        cleanup_run(run_dir, keep_weights_as=os.path.join(LEAKY_DIR, "weights", f"honest_resnet50_seed{seed}.pt"))

if RUN["leaky"]:
    for seed in SEEDS:
        if not leaky_store.has(model="leaky", seed=seed):
            oom_guard(f"[leaky s{seed}]", _leaky_job, seed)
    for seed in SEEDS:
        if not leaky_store.has(model="honest", seed=seed):
            oom_guard(f"[honest s{seed}]", _honest_job, seed)

In [ ]:
# 9.2  summary table ---------------------------------------------------------------------------------------
def pct(v):
    v = [x for x in v if x is not None]
    return fmt_ms(v, 100, 2) if v else "-"

if RUN["leaky"] and leaky_store.rows:
    rows = []
    for name in ("leaky", "honest"):
        R = leaky_store.get(model=name)
        if not R:
            continue
        rows.append([name, len(R), pct([r["acc_test_all"] for r in R]),
                     fmt_ms([r["test_images_contaminated_pct"] for r in R], 1, 1),
                     pct([r.get("acc_test_contaminated") for r in R]) if name == "leaky" else "-",
                     pct([r["acc_test_clean"] for r in R]),
                     pct([r["zs_plantdoc"]["accuracy"] for r in R]), pct([r["zs_fgvc8"]["accuracy"] for r in R]),
                     fmt_ms([r["zs_fgvc8"]["f1_macro"] for r in R], 1, 3)])
    if RUN["sanity"] and sanity.get(backbone="resnet50"):
        g = {r["dataset"]: r for r in sanity.get(backbone="resnet50")}
        rows.append(["paper checkpoint (honest)", 1, f"{g['pv_test']['accuracy']*100:.2f}", "0", "-",
                     f"{g['pv_test']['accuracy']*100:.2f}", f"{g['plantdoc']['accuracy']*100:.2f}",
                     f"{g['fgvc8']['accuracy']*100:.2f}", f"{g['fgvc8']['f1_macro']:.3f}"])
    put_summary("9.2 leaky vs honest",
                "ResNet50, identical training code. 'reported' = accuracy on the model's own test split; "
                "'contaminated/clean' = test images whose physical leaf is / is not in the training split.\n\n" +
                md_table(["model", "seeds", "reported test acc %", "test images contaminated %", "acc contaminated %",
                          "acc clean %", "PlantDoc ZS %", "FGVC8 ZS %", "FGVC8 ZS macro-F1"], rows))

## 10. TRAINING: unsupervised domain adaptation baselines (Reviewer B, Methodology #4)

* **source_only:** ResNet50 (ImageNet init) trained on PlantVillage only. This is the zero-shot reference with this code.
* **CORAL:** plus alignment of the feature covariance between PlantVillage and field batches.
* **DANN:** plus a domain discriminator behind a gradient-reversal layer.

The field images of the 1,600-image pool are used **without labels**; evaluation is on the **same 400-image field test split** as Section 8. So the rows compare directly with the fine-tuning results (which use 80–1,600 *labelled* field images). All runs last `DA_EPOCHS` epochs (the last epoch is used, since no labelled field validation set exists in this setting) and are saved after every epoch.

In [ ]:
# 10.1  CORAL / DANN / source-only ------------------------------------------------------------------------
from torch.autograd import Function
DA_DIR = os.path.join(RESULTS_ROOT, "domain_adaptation")
da_store = ResultStore(os.path.join(DA_DIR, "results.json"))

class GradReverse(Function):
    @staticmethod
    def forward(ctx, x, lambd):
        ctx.lambd = lambd
        return x.view_as(x)
    @staticmethod
    def backward(ctx, g):
        return -ctx.lambd * g, None

class DomainDiscriminator(nn.Module):
    def __init__(self, d, h=256):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d, h), nn.ReLU(True), nn.Dropout(0.3), nn.Linear(h, h // 2), nn.ReLU(True),
                                 nn.Linear(h // 2, 1))
    def forward(self, x):
        return self.net(x).squeeze(1)

def coral_loss(fs, ft):
    # computed in float32 even when mixed precision is on
    ctx = torch.autocast(device_type="cuda", enabled=False) if USE_AMP else contextlib.nullcontext()
    with ctx:
        return _coral(fs.float(), ft.float())

def _coral(fs, ft):
    d = fs.size(1)
    def cov(x):
        x = x - x.mean(0, keepdim=True)
        return x.t() @ x / max(x.size(0) - 1, 1)
    return (cov(fs) - cov(ft)).pow(2).sum() / (4 * d * d)

def run_da(method, seed, lambda_adapt=1.0):
    set_seed(seed)
    run_dir = os.path.join(DA_DIR, "runs", f"{method}_s{seed}")
    os.makedirs(run_dir, exist_ok=True)
    last_p = os.path.join(run_dir, "last.pt")
    model = build_model("resnet50", pretrained=True).to(DEVICE)
    disc = DomainDiscriminator(FEAT_DIM["resnet50"]).to(DEVICE) if method == "dann" else None
    params = list(model.parameters()) + (list(disc.parameters()) if disc is not None else [])
    opt = torch.optim.AdamW(params, lr=1e-4, weight_decay=1e-4)
    scaler = make_scaler()
    start_ep, step = 0, 0
    if os.path.exists(last_p):
        ck = torch_load(last_p)
        model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"]); scaler.load_state_dict(ck["scaler"])
        if disc is not None:
            disc.load_state_dict(ck["disc"])
        start_ep, step = ck["epoch"], ck["step"]
        log(f"  [DA {method} s{seed}] resuming from epoch {start_ep}")
    crit = nn.CrossEntropyLoss(weight=class_weights_for(PV["train"]).to(DEVICE))
    bce = nn.BCEWithLogitsLoss()
    tf = train_transforms(224)
    n_src_batches = len(PV["train"]) // MICRO_BATCH
    total_steps = max(1, DA_EPOCHS * n_src_batches)
    for ep in range(start_ep, DA_EPOCHS):
        model.train()
        if disc is not None:
            disc.train()
        src = make_loader(PV["train"], tf, MICRO_BATCH, shuffle=True, seed=seed * 1000 + ep, drop_last=True)
        tgt = make_loader(FIELD_TRAIN_POOL, tf, MICRO_BATCH, shuffle=True, seed=seed * 2000 + ep, drop_last=True)
        tgt_it = iter(tgt)
        t0, sums = time.time(), defaultdict(float)
        for xs, ys, _ in src:
            try:
                xt = next(tgt_it)[0]
            except StopIteration:
                tgt_it = iter(tgt); xt = next(tgt_it)[0]
            xs, ys, xt = xs.to(DEVICE), ys.to(DEVICE), xt.to(DEVICE)
            with autocast_ctx():
                fs = embed(model, xs)
                cls = crit(logits_from_embedding(model, fs), ys)
                if method == "source_only":
                    adapt = torch.zeros((), device=DEVICE)
                else:
                    ft = embed(model, xt)
                    if method == "coral":
                        adapt = coral_loss(fs, ft)
                    else:
                        p = step / total_steps
                        lambd = 2.0 / (1.0 + math.exp(-10 * p)) - 1.0      # standard DANN schedule 0 -> 1
                        ds_ = disc(GradReverse.apply(fs.float(), lambd))
                        dt_ = disc(GradReverse.apply(ft.float(), lambd))
                        adapt = bce(ds_, torch.zeros_like(ds_)) + bce(dt_, torch.ones_like(dt_))
                loss = cls + lambda_adapt * adapt
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            step += 1
            sums["cls"] += float(cls.item()); sums["adapt"] += float(adapt.item()); sums["n"] += 1
        torch_save_atomic(dict(model=model.state_dict(), opt=opt.state_dict(), scaler=scaler.state_dict(),
                               disc=disc.state_dict() if disc is not None else None, epoch=ep + 1, step=step), last_p)
        print(f"  [DA {method} s{seed}] ep {ep+1}/{DA_EPOCHS} cls={sums['cls']/max(sums['n'],1):.4f} "
              f"adapt={sums['adapt']/max(sums['n'],1):.4f} ({time.time()-t0:.0f}s)")
    return model, last_p

DA_ALLOW_SMALL_GPU = False   # CORAL/DANN need a source AND a target batch in memory at once
if RUN["domain_adapt"] and SMALL_GPU and not DA_ALLOW_SMALL_GPU:
    print("Section 10 skipped: with", MICRO_BATCH, "images per domain the CORAL covariance and the DANN batches are "
          "too small to be meaningful, and 2 batches at once may not fit in memory. Run this section on Colab "
          "(or set DA_ALLOW_SMALL_GPU = True to try anyway).")
elif RUN["domain_adapt"]:
    for method in DA_METHODS:
        for seed in SEEDS:
            if da_store.has(method=method, seed=seed):
                continue
            out = oom_guard(f"[DA {method} s{seed}]", run_da, method, seed)
            if out is None:
                continue
            model, last_p = out
            mf, _ = evaluate(model, FIELD_TEST)
            mp, _ = evaluate(model, PV["test"])
            fe = field_eval(model)
            da_store.add(dict(method=method, seed=seed, epochs=DA_EPOCHS, field_test_acc=mf["accuracy"],
                              field_test_f1=mf["f1_macro"], pv_test_acc=mp["accuracy"],
                              plantdoc_acc=fe["plantdoc"]["accuracy"], fgvc8_eval_acc=fe["fgvc8"]["accuracy"],
                              per_class_f1_field={k: v["f1"] for k, v in mf["per_class"].items()}))
            log(f"[DA {method} s{seed}] field test acc={mf['accuracy']:.4f} F1={mf['f1_macro']:.4f} PV={mp['accuracy']:.4f}")
            os.remove(last_p)
            del model

if RUN["domain_adapt"] and da_store.rows:
    rows = []
    for method in DA_METHODS:
        R = da_store.get(method=method)
        if R:
            rows.append([method, "0 (unlabelled pool)", len(R), fmt_ms([r["field_test_acc"] for r in R], 100, 2),
                         fmt_ms([r["field_test_f1"] for r in R]), fmt_ms([r["pv_test_acc"] for r in R], 100, 2),
                         fmt_ms([r["plantdoc_acc"] for r in R], 100, 2)])
    for init in ("imagenet", "lab"):
        for frac in (0.05, 1.0):
            R = de_store.get(backbone="resnet50", init=init, fraction=frac)
            if R:
                rows.append([f"fine-tune {init}-init (Sec. 8)", R[0]["n_train"], len(R),
                             fmt_ms([r["accuracy"] for r in R], 100, 2), fmt_ms([r["macro_f1"] for r in R]), "-", "-"])
    put_summary("10 domain adaptation vs fine-tuning",
                f"Same FGVC8 field test split (n={len(FIELD_TEST)}).\n\n" +
                md_table(["method", "labelled field images", "seeds", "field acc %", "field macro-F1", "PV test acc %",
                          "PlantDoc acc %"], rows))

## 11. TRAINING: GAN oversampling of cedar apple rust (Reviewer A, Q4)

1. **Generator.** The DCGAN trained earlier (imported from `soic_results (2).zip`) is reused. Set `GAN_TRAIN_NEW = True` to train a new one; it is saved every 25 epochs and resumes.
2. **Filter.** Each synthetic image is kept only if (a) it is not a near-duplicate of a real train/test image (cosine ≥ 0.98 in ResNet50 feature space) **and** (b) the paper classifier calls it cedar apple rust with p ≥ 0.90. Many earlier samples looked like healthy leaves; these would be label noise.
3. **Retrain.** ResNet50 with class weighting (**baseline**) vs class weighting **+ kept synthetic images** (**gan**), same code, 3 seeds each. Tested on PlantVillage (in-domain, saturated) and zero-shot on FGVC8/PlantDoc, where the minority class matters more.

In [ ]:
# 11.1  generator (reuse or train) -----------------------------------------------------------------------
GAN_TRAIN_NEW = False
LATENT, GAN_SIZE = 100, 128

class Generator(nn.Module):
    def __init__(self, nz=LATENT, f=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.ConvTranspose2d(nz, f * 8, 4, 1, 0, bias=False), nn.BatchNorm2d(f * 8), nn.ReLU(True),
            nn.ConvTranspose2d(f * 8, f * 4, 4, 2, 1, bias=False), nn.BatchNorm2d(f * 4), nn.ReLU(True),
            nn.ConvTranspose2d(f * 4, f * 2, 4, 2, 1, bias=False), nn.BatchNorm2d(f * 2), nn.ReLU(True),
            nn.ConvTranspose2d(f * 2, f, 4, 2, 1, bias=False), nn.BatchNorm2d(f), nn.ReLU(True),
            nn.ConvTranspose2d(f, f, 4, 2, 1, bias=False), nn.BatchNorm2d(f), nn.ReLU(True),
            nn.ConvTranspose2d(f, 3, 4, 2, 1, bias=False), nn.Tanh())
    def forward(self, z):
        return self.net(z)

class Discriminator(nn.Module):
    def __init__(self, f=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, f, 4, 2, 1, bias=False), nn.LeakyReLU(0.2, True),
            nn.Conv2d(f, f * 2, 4, 2, 1, bias=False), nn.BatchNorm2d(f * 2), nn.LeakyReLU(0.2, True),
            nn.Conv2d(f * 2, f * 4, 4, 2, 1, bias=False), nn.BatchNorm2d(f * 4), nn.LeakyReLU(0.2, True),
            nn.Conv2d(f * 4, f * 8, 4, 2, 1, bias=False), nn.BatchNorm2d(f * 8), nn.LeakyReLU(0.2, True),
            nn.Conv2d(f * 8, f * 8, 4, 2, 1, bias=False), nn.BatchNorm2d(f * 8), nn.LeakyReLU(0.2, True),
            nn.Conv2d(f * 8, 1, 4, 1, 0, bias=False))
    def forward(self, x):
        return self.net(x).view(-1)

GEN_FINAL = os.path.join(GAN_DIR, "generator_new.pt" if GAN_TRAIN_NEW else "generator_imported.pt")

def train_gan():
    minority = [s for s in PV["train"] if s[1] == CLASSES.index("cedar_apple_rust")]
    tf = T.Compose([T.Resize((GAN_SIZE, GAN_SIZE)), T.RandomHorizontalFlip(), T.RandomVerticalFlip(),
                    T.ToTensor(), T.Normalize([0.5] * 3, [0.5] * 3)])     # no rotation -> no black corners
    G, D = Generator().to(DEVICE), Discriminator().to(DEVICE)
    og = torch.optim.Adam(G.parameters(), lr=2e-4, betas=(0.5, 0.999))
    od = torch.optim.Adam(D.parameters(), lr=2e-4, betas=(0.5, 0.999))
    ck_p, start = os.path.join(GAN_DIR, "gan_train_state.pt"), 0
    if os.path.exists(ck_p):
        ck = torch_load(ck_p)
        G.load_state_dict(ck["G"]); D.load_state_dict(ck["D"]); og.load_state_dict(ck["og"]); od.load_state_dict(ck["od"])
        start = ck["epoch"]
        log(f"GAN resuming at epoch {start}")
    bce = nn.BCEWithLogitsLoss()
    for ep in range(start, GAN_EPOCHS):
        loader = make_loader(minority, tf, min(32, len(minority)), shuffle=True, seed=ep, drop_last=True)
        for real, _, _ in loader:
            real = real.to(DEVICE); b = real.size(0)
            z = torch.randn(b, LATENT, 1, 1, device=DEVICE)
            fake = G(z)
            od.zero_grad()
            ld = bce(D(real), torch.full((b,), 0.9, device=DEVICE)) + bce(D(fake.detach()), torch.zeros(b, device=DEVICE))
            ld.backward(); od.step()
            og.zero_grad()
            lg = bce(D(fake), torch.ones(b, device=DEVICE))
            lg.backward(); og.step()
        if (ep + 1) % 25 == 0 or ep + 1 == GAN_EPOCHS:
            torch_save_atomic(dict(G=G.state_dict(), D=D.state_dict(), og=og.state_dict(), od=od.state_dict(), epoch=ep + 1), ck_p)
            print(f"GAN epoch {ep+1}/{GAN_EPOCHS} loss_D={ld.item():.3f} loss_G={lg.item():.3f}")
    torch_save_atomic(G.state_dict(), GEN_FINAL)

if RUN["gan"]:
    if not os.path.exists(GEN_FINAL):
        if GAN_TRAIN_NEW or FAST_DEV_RUN:
            GEN_FINAL = os.path.join(GAN_DIR, "generator_new.pt")
            if not os.path.exists(GEN_FINAL):
                train_gan()
        else:
            raise FileNotFoundError("generator_imported.pt not found - put 'soic_results (2).zip' next to the notebook "
                                    "(Section 2.3 imports it) or set GAN_TRAIN_NEW = True")
    print("generator:", GEN_FINAL)

In [ ]:
# 11.2  generate + filter --------------------------------------------------------------------------------
GEN_DIR = os.path.join(GAN_DIR, "generated_" + os.path.splitext(os.path.basename(GEN_FINAL))[0])
FILTER_JSON = os.path.join(GAN_DIR, "filter_report.json")
if RUN["gan"]:
    os.makedirs(GEN_DIR, exist_ok=True)
    existing = sorted(glob.glob(os.path.join(GEN_DIR, "gen_*.png")))
    if len(existing) < GAN_N_GENERATE:
        G = Generator().to(DEVICE)
        G.load_state_dict(torch_load(GEN_FINAL)); G.eval()
        g = torch.Generator().manual_seed(2026)
        with torch.no_grad():
            z = torch.randn(GAN_N_GENERATE, LATENT, 1, 1, generator=g)
            for i in range(0, GAN_N_GENERATE, 32):
                imgs = (G(z[i:i + 32].to(DEVICE)).cpu() * 0.5 + 0.5).clamp(0, 1)
                for k, im in enumerate(imgs):
                    Image.fromarray((im.permute(1, 2, 0).numpy() * 255).astype(np.uint8)).save(
                        os.path.join(GEN_DIR, f"gen_{i+k:04d}.png"))
        existing = sorted(glob.glob(os.path.join(GEN_DIR, "gen_*.png")))
    gen_samples = [(p, CLASSES.index("cedar_apple_rust")) for p in existing[:GAN_N_GENERATE]]
    clf = paper_classifier("resnet50")
    cedar = CLASSES.index("cedar_apple_rust")
    f_gen = extract_embeddings(clf, gen_samples)
    f_tr = extract_embeddings(clf, [s for s in PV["train"] if s[1] == cedar])
    f_te = extract_embeddings(clf, [s for s in PV["test"] if s[1] == cedar])
    sim_tr, sim_te = cos_matrix(f_gen, f_tr).max(1), cos_matrix(f_gen, f_te).max(1)
    probs = predict(clf, gen_samples)["probs"]
    report = []
    for (p, _), a, b, pr in zip(gen_samples, sim_tr, sim_te, probs):
        near_dup = bool(a >= GAN_SIM_THRESH or b >= GAN_SIM_THRESH)
        confident = bool(pr[cedar] >= GAN_MIN_CONF)
        report.append(dict(path=p, sim_train=float(a), sim_test=float(b), p_cedar=float(pr[cedar]),
                           predicted=CLASSES[int(pr.argmax())], near_duplicate=near_dup, confident=confident,
                           keep=(not near_dup) and confident))
    save_json(report, FILTER_JSON)
    kept = [r for r in report if r["keep"]]
    pred_counts = Counter(r["predicted"] for r in report)
    put_summary("11.2 GAN filter",
                f"{len(report)} synthetic images: near-duplicates {sum(r['near_duplicate'] for r in report)}, "
                f"classified as cedar rust with p>={GAN_MIN_CONF}: {sum(r['confident'] for r in report)}, kept {len(kept)}.\n"
                f"Classifier's predicted class for the synthetic images: {dict(pred_counts)}. "
                f"Max similarity to a real training image: median {np.median(sim_tr):.3f}, max {sim_tr.max():.3f}.")
    fig, axes = plt.subplots(2, 8, figsize=(14, 4))
    rej = [r for r in report if not r["keep"]]
    for row, (title, R) in enumerate([("kept", kept), ("rejected", rej)]):
        for j in range(8):
            ax = axes[row, j]; ax.axis("off")
            if j < len(R):
                ax.imshow(Image.open(R[j]["path"]))
                ax.set_title(f"{R[j]['predicted'][:10]} {R[j]['p_cedar']:.2f}", fontsize=7)
        axes[row, 0].annotate(title, xy=(-0.12, 0.5), xycoords="axes fraction", rotation=90,
                              va="center", ha="right", fontsize=10)
    save_fig(fig, "fig_gan_kept_vs_rejected"); plt.show()

In [ ]:
# 11.3  baseline vs +GAN retraining --------------------------------------------------------------------
gan_store = ResultStore(os.path.join(GAN_DIR, "retrain_results.json"))
if RUN["gan"]:
    kept_samples = [(r["path"], CLASSES.index("cedar_apple_rust")) for r in load_json(FILTER_JSON, []) if r["keep"]]
    for cond in ("baseline", "gan"):
        for seed in SEEDS:
            if gan_store.has(condition=cond, seed=seed):
                continue
            def _gan_job(cond, seed):
                set_seed(seed)
                train = PV["train"] + (kept_samples if cond == "gan" else [])
                run_dir = os.path.join(GAN_DIR, "runs", f"{cond}_s{seed}")
                model = build_model("resnet50", pretrained=True)
                model, info = fit_two_phase(model, train, PV["val"], run_dir, seed, tag=f"[GAN {cond} s{seed}]")
                mp, _ = evaluate(model, PV["test"])
                mf, _ = evaluate(model, FGVC8_EVAL)
                md_, _ = evaluate(model, PLANTDOC, PLANTDOC_PRESENT)
                gan_store.add(dict(condition=cond, seed=seed, n_synthetic=len(train) - len(PV["train"]),
                                   pv_acc=mp["accuracy"], pv_f1=mp["f1_macro"], pv_cedar_f1=mp["per_class"]["cedar_apple_rust"]["f1"],
                                   fgvc8_acc=mf["accuracy"], fgvc8_f1=mf["f1_macro"],
                                   fgvc8_cedar_recall=mf["per_class"]["cedar_apple_rust"]["recall"],
                                   fgvc8_cedar_f1=mf["per_class"]["cedar_apple_rust"]["f1"],
                                   plantdoc_acc=md_["accuracy"], plantdoc_cedar_recall=md_["per_class"]["cedar_apple_rust"]["recall"]))
                log(f"[GAN {cond} s{seed}] PV acc={mp['accuracy']:.4f} FGVC8 acc={mf['accuracy']:.4f} "
                    f"FGVC8 cedar recall={mf['per_class']['cedar_apple_rust']['recall']:.3f}")
                cleanup_run(run_dir)

            oom_guard(f"[GAN {cond} s{seed}]", _gan_job, cond, seed)
    rows = []
    for cond in ("baseline", "gan"):
        R = gan_store.get(condition=cond)
        if R:
            rows.append([cond, R[0]["n_synthetic"], len(R), fmt_ms([r["pv_acc"] for r in R], 100, 2),
                         fmt_ms([r["pv_cedar_f1"] for r in R]), fmt_ms([r["fgvc8_acc"] for r in R], 100, 2),
                         fmt_ms([r["fgvc8_cedar_recall"] for r in R]), fmt_ms([r["plantdoc_acc"] for r in R], 100, 2),
                         fmt_ms([r["plantdoc_cedar_recall"] for r in R])])
    if rows:
        put_summary("11.3 GAN oversampling", md_table(["condition", "synthetic imgs", "seeds", "PV acc %", "PV cedar F1",
                                                       "FGVC8 ZS acc %", "FGVC8 cedar recall", "PlantDoc ZS acc %",
                                                       "PlantDoc cedar recall"], rows))

## 12. (Optional) TRAINING: which augmentation group helps the domain gap? (Reviewer A, Q2)

ResNet50 trained with *none*, *geometric only*, *photometric only*, *occlusion only* and the *full* policy. Each is tested in-domain and zero-shot on both field sets. Switch on with `RUN["aug_ablation"] = True`.

In [ ]:
# 12.1  augmentation-group ablation ----------------------------------------------------------------------
AUG_GROUPS = {"none": (), "geometric": ("geometric",), "photometric": ("photometric",),
              "occlusion": ("occlusion",), "full": ("geometric", "photometric", "occlusion")}
AUG_DIR = os.path.join(RESULTS_ROOT, "aug_ablation")
aug_store = ResultStore(os.path.join(AUG_DIR, "results.json"))
if RUN["aug_ablation"]:
    for g, groups in AUG_GROUPS.items():
        for seed in AUG_SEEDS:
            if aug_store.has(policy=g, seed=seed):
                continue
            def _aug_job(g, groups, seed):
                set_seed(seed)
                run_dir = os.path.join(AUG_DIR, "runs", f"{g}_s{seed}")
                model = build_model("resnet50", pretrained=True)
                model, _ = fit_two_phase(model, PV["train"], PV["val"], run_dir, seed,
                                         train_tf=train_transforms(224, groups), tag=f"[aug {g} s{seed}]")
                mp, _ = evaluate(model, PV["test"])
                fe = field_eval(model)
                aug_store.add(dict(policy=g, seed=seed, pv_acc=mp["accuracy"], plantdoc_acc=fe["plantdoc"]["accuracy"],
                                   fgvc8_acc=fe["fgvc8"]["accuracy"], fgvc8_f1=fe["fgvc8"]["f1_macro"]))
                cleanup_run(run_dir)
            oom_guard(f"[aug {g} s{seed}]", _aug_job, g, groups, seed)
    rows = [[g, len(aug_store.get(policy=g)), fmt_ms([r["pv_acc"] for r in aug_store.get(policy=g)], 100, 2),
             fmt_ms([r["plantdoc_acc"] for r in aug_store.get(policy=g)], 100, 2),
             fmt_ms([r["fgvc8_acc"] for r in aug_store.get(policy=g)], 100, 2)] for g in AUG_GROUPS if aug_store.get(policy=g)]
    put_summary("12 augmentation ablation", md_table(["policy", "seeds", "PV acc %", "PlantDoc ZS %", "FGVC8 ZS %"], rows))

## 13. (Optional, very heavy) TRAINING: five seeds for every backbone (Reviewer B)

Re-trains the in-domain classifiers with 5 seeds (25 runs). Only needed if you decide to report 5 seeds instead of justifying 3 seeds.

In [ ]:
# 13.1  five-seed in-domain sweep -------------------------------------------------------------------------
FIVE_DIR = os.path.join(RESULTS_ROOT, "five_seeds")
five_store = ResultStore(os.path.join(FIVE_DIR, "results.json"))
if RUN["five_seeds"]:
    for bb in ALL_BACKBONES:
        for seed in FIVE_SEEDS:
            if five_store.has(backbone=bb, seed=seed):
                continue
            def _five_job(bb, seed):
                set_seed(seed)
                run_dir = os.path.join(FIVE_DIR, "runs", f"{bb}_s{seed}")
                model = build_model(bb, pretrained=True)
                model, info = fit_two_phase(model, PV["train"], PV["val"], run_dir, seed, tag=f"[5seeds {bb} s{seed}]")
                m, _ = evaluate(model, PV["test"])
                five_store.add(dict(backbone=bb, seed=seed, accuracy=m["accuracy"], f1_macro=m["f1_macro"],
                                    roc_auc=m.get("roc_auc_ovr"), best_epoch=info["best_epoch"]))
                cleanup_run(run_dir)
            oom_guard(f"[5seeds {bb} s{seed}]", _five_job, bb, seed)
    rows = [[bb, len(five_store.get(backbone=bb)), fmt_ms([r["accuracy"] for r in five_store.get(backbone=bb)], 100, 2),
             fmt_ms([r["f1_macro"] for r in five_store.get(backbone=bb)], 1, 4)] for bb in ALL_BACKBONES if five_store.get(backbone=bb)]
    put_summary("13 five seeds", md_table(["backbone", "seeds", "accuracy %", "macro-F1"], rows))

## 14. One summary file for writing the revision

Collects every section's table into `result_new/ALL_RESULTS_SUMMARY.md`. Send this file (and the `figures` folder) back to Claude to update the manuscript and the response letter.

In [ ]:
# 14.1  write ALL_RESULTS_SUMMARY.md -------------------------------------------------------------------
summ = load_json(SUMMARY_PATH, {})
def _order(k):
    m = re.match(r"(\d+)(?:\.(\d+))?", k)
    return (int(m.group(1)), int(m.group(2) or 0)) if m else (99, 0)
lines = [f"# SOIC 4492 revision - experiment results", "",
         f"generated {time.strftime('%Y-%m-%d %H:%M')} on {platform.node()} ({DEVICE}), FAST_DEV_RUN={FAST_DEV_RUN}", ""]
for k in sorted(summ, key=_order):
    lines += [f"## {k}", "", summ[k], ""]
out = os.path.join(RESULTS_ROOT, "ALL_RESULTS_SUMMARY.md")
with open(out, "w", encoding="utf-8") as f:
    f.write("\n".join(lines))
print("written:", out)
if OOM_SKIPPED:
    print("Skipped because the GPU ran out of memory (run these on a bigger GPU):", OOM_SKIPPED)
print("figures:", sorted(os.listdir(FIG_DIR)))